In [ ]:
### Local Runtime ###
# run in mini forge prompt
# jupyter notebook --NotebookApp.allow_origin='https://colab.research.google.com' --port=8888 --NotebookApp.port_retries=0 --NotebookApp.allow_credentials=True

In [ ]:
### Connect Jupyter Console ###
%connect_info

connenction_info = {"key":"ea964a7f-2ec1-42ab-a9bb-b1d78c6453c2","signature_scheme":"hmac-sha256","transport":"tcp","ip":"127.0.0.1","hb_port":9005,"control_port":9006,"shell_port":9007,"stdin_port":9008,"iopub_port":9009,"kernel_name":"python3135jvsc74a57bd088bd71113c8908768bb24a9b4458ff77a78e5cb2e14f124bdf1bb088b5add581"}


import json
with open("kernel.json", "w") as f:
    json.dump(connenction_info, f, indent=4)

# !jupyter console --existing src/transformer_tests/kernel.json

In [1]:
# =========================
# Colab Tiny Transformer Trainer for Simple English Wikipedia
# Model: Llama-inspired, 4L, 256H, 4K vocab, RoPE, 256 window
# =========================

# --------- 0. Environment Setup ---------
# !pip install -U torch tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# from datasets import load_dataset
# from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from tqdm import tqdm
import numpy as np
from datetime import datetime

## Colab - mount Google Drive
# from google.colab import files, drive
# drive.mount('/content/drive')

## Set models path
# Colab - Google Drive
# models_path = '/content/drive/My Drive/LanguageDynamics/models/'

# Remote (SSH) - Linux path
models_path = '/home/galk/LanguageDynamics/models'

# Local - Windows path
# models_path = 'C:/Users/Gankl/PycharmProjects/LanguageDynamics/models'

In [2]:
# --------- 1. Config ---------
### Language Modeling ###
# CONFIG = {
#     'mode': 'LM',
#     'n_layers': 2,
#     'n_heads': 2,
#     'embed_dim': 32,
#     'ffn_dim': 256,
#     'vocab': ['(', ')', '[', ']', '<BOS>', '<EOS>', '<SOS>', '<CLS>'],
#     'context_window': 32,
#     'max_depth': 4,
#     'min_length': 4,
#     'max_length': 50,
#     'batch_size': 128,
#     'epochs': 500,
#     'lr': 1e-4,
#     'save_every': 1,
#     'checkpoint_path': '/home/galk/LanguageDynamics/models/tiny_LM_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_maxdepth_4_maxlen_50_date_120825_1039_trial_1/ckpt_epoch145.pt',
#     # 'checkpoint_path': None,
#     'device': 'cuda' if torch.cuda.is_available() else 'cpu'
# }

# ## Autoencoder ###
# CONFIG = {
#     'mode': 'AE',
#     'n_layers': 2,
#     'n_heads': 2,
#     'embed_dim': 32,
#     'ffn_dim': 256,
#     'vocab': ['(', ')', '[', ']', '<BOS>', '<EOS>', '<SOS>', '<CLS>'],
#     'context_window': 32,
#     'latent_dim': 32,
#     'n_latents': 4,
#     'max_depth': 4,
#     'min_length': 4,
#     'max_length': 50,
#     'batch_size': 512,
#     'epochs': 500,
#     'lr': 3e-4,
#     'save_every': 1,
#     # 'checkpoint_path': '/content/drive/MyDrive/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_100825_0456_trial_1/ckpt_epoch100.pt',
#     # 'checkpoint_path': 'C:/Users/Gankl/PycharmProjects/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_16_ffn_dim_128_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_080825_1626_trial_1/ckpt_epoch109.pt',
#     'checkpoint_path': None,
#     'device': 'cuda' if torch.cuda.is_available() else 'cpu'
# }

## Koopman Autoencoder ###
CONFIG = {
    'mode': 'KAE',
    'n_layers': 2,
    'n_heads': 2,
    'embed_dim': 32,
    'ffn_dim': 256,
    'vocab': ['(', ')', '[', ']', '<BOS>', '<EOS>', '<SOS>', '<CLS>'],
    'context_window': 32,
    'latent_dim': 32,
    'n_latents': 4,
    'n_diagonals': 5,  # Number of diagonals in the Koopman operator
    'max_depth': 4,
    'min_length': 4,
    'max_length': 50,
    'batch_size': 64,
    'epochs': 500,
    'lr': 3e-4,
    'save_every': 1,
    # 'checkpoint_path': '/content/drive/MyDrive/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_100825_0456_trial_1/ckpt_epoch100.pt',
    # 'checkpoint_path': 'C:/Users/Gankl/PycharmProjects/LanguageDynamics/models/tiny_AE_dyck2_layers_2_embed_16_ffn_dim_128_context_window_32_latent_32_n_latents_4_maxdepth_4_maxlen_50_date_080825_1626_trial_1/ckpt_epoch109.pt',
    # 'checkpoint_path': '/home/galk/LanguageDynamics/models/tiny_KAE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_n_diagonals_5_maxdepth_4_maxlen_50_date_150825_2321_trial_1/ckpt_epoch170.pt',
    'checkpoint_path': None,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
if CONFIG['mode'] == 'AE':
    CONFIG['model_save_prefix'] = f"tiny_{CONFIG['mode']}_dyck2_layers_{CONFIG['n_layers']}_embed_{CONFIG['embed_dim']}_ffn_dim_{CONFIG['ffn_dim']}_context_window_{CONFIG['context_window']}_latent_{CONFIG['latent_dim']}_n_latents_{CONFIG['n_latents']}_maxdepth_{CONFIG['max_depth']}_maxlen_{CONFIG['max_length']}_date_{current_date_ddmmyy}"
if CONFIG['mode'] == 'KAE':
    CONFIG['model_save_prefix'] = f"tiny_{CONFIG['mode']}_dyck2_layers_{CONFIG['n_layers']}_embed_{CONFIG['embed_dim']}_ffn_dim_{CONFIG['ffn_dim']}_context_window_{CONFIG['context_window']}_latent_{CONFIG['latent_dim']}_n_latents_{CONFIG['n_latents']}_n_diagonals_{CONFIG['n_diagonals']}_maxdepth_{CONFIG['max_depth']}_maxlen_{CONFIG['max_length']}_date_{current_date_ddmmyy}"
elif CONFIG['mode'] == 'LM':
    CONFIG['model_save_prefix'] = f"tiny_{CONFIG['mode']}_dyck2_layers_{CONFIG['n_layers']}_embed_{CONFIG['embed_dim']}_ffn_dim_{CONFIG['ffn_dim']}_context_window_{CONFIG['context_window']}_maxdepth_{CONFIG['max_depth']}_maxlen_{CONFIG['max_length']}_date_{current_date_ddmmyy}"

## Overide model save prefix
# CONFIG['model_save_prefix'] = '/home/galk/LanguageDynamics/models/tiny_KAE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_n_diagonals_5_maxdepth_4_maxlen_50_date_150825_2321'

CONFIG['vocab_size'] = len(CONFIG['vocab'])

print(f"Using device: {CONFIG['device']}")
print("Vocabulary:", CONFIG['vocab'])
print(f"{CONFIG['model_save_prefix']}")

Using device: cuda
Vocabulary: ['(', ')', '[', ']', '<BOS>', '<EOS>', '<SOS>', '<CLS>']
tiny_KAE_dyck2_layers_2_embed_32_ffn_dim_256_context_window_32_latent_32_n_latents_4_n_diagonals_5_maxdepth_4_maxlen_50_date_180825_0734


In [3]:
# --------- 2. Dyck-2 Dataset Generation ---------
# For reproducibility, we generate a Dyck-2 dataset on the fly

import random
from typing import List, Tuple

def generate_dyck2_sequence(max_depth: int = 3,
                          min_len: int = 10,
                          max_len: int = 10) -> List[str]:
    """
    Generates a single valid Dyck-2 string with bounded depth.

    Args:
        max_depth (int): Maximum nesting depth allowed
        min_len (int): Minimum sequence length (must be even)
        max_len (int): Maximum sequence length (must be even)

    Returns:
        List[str]: A valid Dyck-2 sequence as a list of characters

    Raises:
        ValueError: If parameters are invalid
    """
    # Validate parameters
    if min_len < 2 or max_len < min_len:
        raise ValueError("Invalid length parameters")
    if max_depth < 1:
        raise ValueError("max_depth must be positive")

    # Ensure even lengths
    min_len = (min_len + 1) & ~1  # Round up to even
    max_len = max_len & ~1  # Round down to even

    pairs: List[Tuple[str, str]] = [('(', ')'), ('[', ']')]
    seq: List[str] = []
    stack: List[str] = []
    target_len = random.randint(min_len, max_len)

    remaining_space = target_len - len(seq)

    while len(seq) < target_len or stack:
        can_open = (len(seq) < target_len - len(stack) and
                   len(stack) < max_depth)
        can_close = bool(stack)

        # Calculate if we must close to meet constraints
        must_close = (len(stack) + remaining_space == target_len or
                     len(seq) + len(stack) == target_len)

        if can_close and (must_close or (not can_open) or random.random() > 0.5):
            # Close bracket
            seq.append(stack.pop())
        elif can_open:
            # Open new bracket
            pair = random.choice(pairs)
            seq.append(pair[0])
            stack.append(pair[1])
        else:
            # Cannot proceed - restart generation
            return generate_dyck2_sequence(max_depth, min_len, max_len)

        remaining_space = target_len - len(seq)

    return seq

# def generate_dyck2_dataset(n_samples: int = 10000,
#                           **kwargs) -> List[List[str]]:
#     """
#     Generates multiple Dyck-2 sequences.

#     Args:
#         n_samples (int): Number of sequences to generate
#         **kwargs: Parameters passed to generate_dyck2_sequence

#     Returns:
#         List[List[str]]: List of Dyck-2 sequences
#     """
#     return [generate_dyck2_sequence(**kwargs) for _ in tqdm(range(n_samples))]

def generate_dyck2_dataset(n_samples: int = 10000, seen = None, **kwargs) -> List[List[str]]:
    """
    Generates multiple *unique* Dyck-2 sequences.

    Args:
        n_samples (int): Number of unique sequences to generate
        **kwargs: Parameters passed to generate_dyck2_sequence

    Returns:
        List[List[str]]: List of unique Dyck-2 sequences
    """
    if seen is None:
        seen = set()
    dataset = []

    pbar = tqdm(total=n_samples)
    while len(dataset) < n_samples:
        seq = generate_dyck2_sequence(**kwargs)
        seq_str = ''.join(seq)
        if seq_str not in seen:
            seen.add(seq_str)
            dataset.append(seq)
            pbar.update(1)
    pbar.close()

    return dataset, seen


def is_valid_dyck2(sequence: List[str], max_depth: int = None) -> bool:
    """Validates if a sequence is a valid Dyck-2 string."""
    stack = []
    pairs = {'(': ')', '[': ']'}
    max_seen_depth = 0

    for char in sequence:
        if char in '([':
            stack.append(char)
            max_seen_depth = max(max_seen_depth, len(stack))
            if max_depth and max_seen_depth > max_depth:
                return False
        else:
            if not stack or pairs[stack.pop()] != char:
                return False

    return len(stack) == 0

In [4]:
# Generate datasets
print("Generating Dyck-2 dataset...")

train_seqs, seen_train = generate_dyck2_dataset(
    n_samples=200000,
    max_depth=CONFIG['max_depth'],
    min_len=CONFIG['min_length'],
    max_len=CONFIG['max_length']
)
val_seqs, seen_val = generate_dyck2_dataset(
    n_samples=30000,
    seen=seen_train,
    max_depth=CONFIG['max_depth'],
    min_len=CONFIG['min_length'],
    max_len=CONFIG['max_length']
)

# Validation checks
example_seq = train_seqs[0]
print()
print(f"Example sequence: {''.join(example_seq)}")
print(f"Length: {len(example_seq)}")

Generating Dyck-2 dataset...


100%|██████████| 30000/30000 [00:00<00:00, 37328.19it/s]


Example sequence: [][]()[]()(()())[[([])([]())[()]]]
Length: 34


In [5]:
# --------- 3. Tokenizer (fixed vocabulary) ---------
# Dyck-2 is synthetic, so we use a fixed vocab and simple mapping
from transformers import PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel

token2id = {tok: idx for idx, tok in enumerate(CONFIG['vocab'])}
id2token = {idx: tok for tok, idx in token2id.items()}
# pad_id = token2id['<PAD>']
pad_id = 100
bos_id = token2id['<BOS>']
eos_id = token2id['<EOS>']
if CONFIG['mode'] == 'AE' or CONFIG['mode'] == 'KAE':
    sos_id = token2id['<SOS>']
    cls_id = token2id['<CLS>']

# Step 1: Create the base tokenizer
word_level = WordLevel(vocab=token2id, unk_token=None)  # explicitly no unk token
tokenizer_backend = Tokenizer(word_level)

# Step 3: Wrap in HuggingFace PreTrainedTokenizerFast
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer_backend,
    bos_token="<BOS>",
    eos_token="<EOS>",
    unk_token=None,
    pad_token=None
)

def encode(seq):
    # Map tokens to ids, add BOS at start, EOS at end, pad to context_window
    ids = [bos_id] + [token2id[tok] for tok in seq] + [eos_id]  # encode with BOS-EOS tokens
    # ids = [bos_id] + [hf_tokenizer.encode(tok)[0] for tok in seq] + [eos_id]  # encode with BOS-EOS tokens
    # ids = [token2id[tok] for tok in seq]  # encode without special tokens
    return ids

def decode(ids):
    # Map ids back to tokens, remove PAD/BOS/EOS
    return [id2token[i] for i in ids if id2token[i] not in ['<PAD>', '<BOS>', '<EOS>', '<SOS>', '<CLS>']]

def prepare_data_block(seqs: List, block_size):
    all_ids = []
    for seq in seqs:
        all_ids += encode(seq)
    n_blocks = len(all_ids) // block_size
    data_blocks = np.array(all_ids[:n_blocks * block_size]).reshape(n_blocks, block_size)
    return data_blocks

train_blocks = prepare_data_block(train_seqs, CONFIG['context_window'])
val_blocks = prepare_data_block(val_seqs, CONFIG['context_window'])

/home/galk/miniforge3/envs/master-dl-base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# --------- 5. Dataset and DataLoader ---------
class DyckDataset(Dataset):
    def __init__(self, data_blocks):
        self.data = torch.tensor(data_blocks, dtype=torch.long)

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        x = self.data[idx, :-1]
        y = self.data[idx, 1:]
        return x, y

class DyckAutoencoderDataset(Dataset):
    def __init__(self, data_blocks):
        self.data = torch.tensor(data_blocks, dtype=torch.long)

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        x = self.data[idx]
        y = self.data[idx]
        return x, y

# Create datasets
train_dataset = DyckDataset(train_blocks) if CONFIG['mode'] == 'LM' else DyckAutoencoderDataset(train_blocks)
val_dataset = DyckDataset(val_blocks) if CONFIG['mode'] == 'LM' else DyckAutoencoderDataset(val_blocks)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,  # No need to shuffle validation data
    drop_last=True
)

# Verify the split
print("\nDataLoader Info:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")


DataLoader Info:
Training batches: 3333
Validation batches: 510


In [7]:
# --------- 6. Transformer Model (Llama-inspired, RoPE) ---------
# RoPE implementation (from Llama paper and HF)
def apply_rope(x, base=10000.0, seq_dim=1):
    # x: [batch, seq, n_heads, head_dim]
    # RoPE mixes head_dim pairs
    batch, seq, n_heads, head_dim = x.size()
    half_dim = head_dim // 2
    pos = torch.arange(seq, dtype=torch.float32, device=x.device)
    idx = torch.arange(half_dim, dtype=torch.float32, device=x.device)
    freq = torch.exp(-math.log(base) * idx / half_dim)
    angles = pos[:, None] * freq[None, :]
    cos, sin = torch.cos(angles), torch.sin(angles)
    x1, x2 = x[..., :half_dim], x[..., half_dim:]
    x_rope = torch.cat([x1 * cos[None, :, None, :] - x2 * sin[None, :, None, :],
                        x1 * sin[None, :, None, :] + x2 * cos[None, :, None, :]], dim=-1)
    return x_rope

def generate_sinusoidal_embeddings(n_latents, embed_dim):
    pe = torch.zeros(n_latents, embed_dim)
    position = torch.arange(0, n_latents, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe.unsqueeze(0) # [1, n_latents, embed_dim]

class RoPEMultiheadAttention(nn.Module):
    def __init__(self, embed_dim, n_heads, causal_mask=True, dropout=0.05):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.causal_mask = causal_mask

    def forward(self, x):
        # x: [batch, seq, embed_dim]
        B, S, E = x.shape

        # Create causal mask
        if self.causal_mask:
            causal_mask = torch.triu(torch.ones(S, S, device=x.device) * float('-inf'), diagonal=1)

        qkv = self.qkv_proj(x)                # [B, S, 3E]
        q, k, v = qkv.chunk(3, dim=-1)        # [B, S, E] each

        # reshape for multihead: [B, S, n_heads, head_dim]
        def split_heads(t):
            return t.view(B, S, self.n_heads, self.head_dim)
        q, k, v = map(split_heads, (q, k, v))

        # Apply RoPE to q and k
        q, k = apply_rope(q), apply_rope(k)

        # [B, n_heads, S, head_dim]
        q, k, v = [x.permute(0,2,1,3) for x in (q,k,v)]

        # Scaled dot-product attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        if self.causal_mask:
            attn_weights = attn_weights + causal_mask[None, None, :, :]  ## MASK ADDED BY CLAUDE
        attn_weights = self.dropout(attn_weights.softmax(dim=-1))  ## DROPOUT ADDED BY CLAUDE

        attn_output = torch.matmul(attn_weights, v)   # [B, n_heads, S, head_dim]
        attn_output = attn_output.permute(0,2,1,3).contiguous().view(B, S, E)
        return self.out_proj(attn_output)

class MultiheadCrossAttention(nn.Module):
    def __init__(self, embed_dim, latent_dim, n_latents, n_heads, dropout=0.03):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(latent_dim, embed_dim * n_latents, bias=False)
        self.v_proj = nn.Linear(latent_dim, embed_dim * n_latents, bias=False)
        # self.qkv_proj = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.n_latents = n_latents

        # Pre-compute and store sinusoidal positional encodings
        pe = generate_sinusoidal_embeddings(self.n_latents, embed_dim)
        self.register_buffer('positional_encodings', pe)

    def forward(self, x, z):
        # x: [batch, seq, embed_dim]
        # z: [batch, 1, latent_dim]
        B, S, E = x.shape

        q = self.q_proj(x)                # [B, S, E]
        k = self.k_proj(z)                # [B, 1, E * n_latents]
        v = self.v_proj(z)                # [B, 1, E * n_latents]

        k = k.view(B, self.n_latents, E)  # [B, n_latents, E]
        v = v.view(B, self.n_latents, E)  # [B, n_latents, E]
        # kv = kv.view(B, self.n_latents, E * 2)
        # k, v = kv.chunk(2, dim=-1)        # [B, n_latents, E] each
        # q, k, v = qkv.chunk(3, dim=-1)        # [B, S, E] each

        # Add positional encodings to the Keys and Values
        # The positional_encodings tensor is [1, n_latents, embed_dim] and will broadcast
        k = k + self.positional_encodings
        v = v + self.positional_encodings

        # reshape for multihead: [B, S, n_heads, head_dim]
        def split_heads(t):
            return t.view(B, t.size(1), self.n_heads, self.head_dim)
        q, k, v = map(split_heads, (q, k, v))

        # [B, n_heads, S, head_dim]
        q, k, v = [x.permute(0,2,1,3) for x in (q,k,v)]

        # Scaled dot-product attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.head_dim)
        attn_weights = self.dropout(attn_weights.softmax(dim=-1))  ## DROPOUT ADDED BY CLAUDE

        attn_output = torch.matmul(attn_weights, v)   # [B, n_heads, S, head_dim]
        attn_output = attn_output.permute(0,2,1,3).contiguous().view(B, S, E)
        return self.out_proj(attn_output)

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, ffn_dim, dropout=0.03, causal_mask=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = RoPEMultiheadAttention(embed_dim, n_heads, causal_mask=causal_mask)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class TransformerDecoderBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, ffn_dim, latent_dim, n_latents, dropout=0.03):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = RoPEMultiheadAttention(embed_dim, n_heads, causal_mask=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.cross_attention = MultiheadCrossAttention(embed_dim, latent_dim, n_latents, n_heads)
        self.ln3 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, embed_dim)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, z):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.cross_attention(self.ln2(x), z))
        x = x + self.dropout(self.ffn(self.ln3(x)))
        return x

class TinyLlamaTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, n_layers, n_heads, ffn_dim, context_window):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = None  # RoPE only
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.head.weight = self.embed.weight  # Optional head-embed weight tying
        self.context_window = context_window

    def forward(self, x, return_internals=False):
        """
        Forward pass supporting token indices, probability distributions over tokens, or embedded vectors.

        Args:
            x (Tensor):
                - [B, T] if token indices (int)
                - [B, T, vocab_size] if probability distributions
                - [B, T, embed_dim] if pre-computed embeddings
            return_internals (bool): whether to return intermediate embeddings

        Returns:
            logits or (logits, initial_embeddings, final_embeddings)
        """
        B = x.size(0)
        T = x.size(1)
        E = self.embed.embedding_dim
        V = self.embed.num_embeddings

        if T > self.context_window:
            raise ValueError(f"Input sequence length {T} exceeds context window {self.context_window}")

        # Case 1: Token indices [B, T]
        if x.ndim == 2:
            x = self.embed(x)  # [B, T, E]

        # Case 2: Probability distributions over vocabulary [B, T, V]
        elif x.ndim == 3 and x.shape[2] == V:
            # Ensure it sums to 1 along the vocab dimension
            if not torch.allclose(x.sum(dim=2), torch.ones(B, T, device=x.device), atol=1e-4):
                raise ValueError("Input probabilities must sum to 1 along vocab dimension.")
            x = torch.matmul(x, self.embed.weight)  # [B, T, E]

        # Case 3: Precomputed embeddings [B, T, E]
        elif x.ndim == 3 and x.shape[2] == E:
            pass  # already in embedded space

        else:
            raise ValueError(f"Unrecognized input shape: {x.shape}")

        initial_embeddings = x.clone()

        for layer in self.layers:
            x = layer(x)

        x = self.ln_f(x)
        final_embeddings = x.clone()
        logits = self.head(x)

        if return_internals:
            return logits, initial_embeddings, final_embeddings
        else:
            return logits

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id, embed_dropout=0.03, normalize_before_projection=True):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.embed_dropout = nn.Dropout(embed_dropout)
        self.layers = nn.ModuleList([
            TransformerBlock(embed_dim, n_heads, ffn_dim, causal_mask=False) for _ in range(n_layers)
        ])
        self.latent_projection = nn.Linear(embed_dim, latent_dim, bias=False)
        self.layer_norm = nn.LayerNorm(embed_dim)
        self.context_window = context_window
        self.cls_id = cls_id
        self.normalize_before_projection = normalize_before_projection


    def forward(self, x, return_internals=True):
        """
        Forward pass supporting token indices, probability distributions over tokens, or embedded vectors.

        Args:
            x (Tensor):
                - [B, T] if token indices (int)
                - [B, T, vocab_size] if probability distributions
                - [B, T, embed_dim] if pre-computed embeddings
            return_internals (bool): whether to return intermediate embeddings

        Returns:
            logits or (logits, initial_embeddings, final_embeddings)
        """
        B = x.size(0)
        T = x.size(1)
        E = self.embed.embedding_dim
        V = self.embed.num_embeddings

        if T > self.context_window:
            raise ValueError(f"Input sequence length {T} exceeds context window {self.context_window}")

        # Case 1: Token indices [B, T]
        if x.ndim == 2:
            # prepend [CLS] token
            x = torch.concat([torch.full((B, 1), self.cls_id, dtype=torch.long, device=x.device), x], dim=1)  # [B, T+1]
            x = self.embed(x)  # [B, T+1, E]

        # Case 2: Probability distributions over vocabulary [B, T, V]
        elif x.ndim == 3 and x.shape[2] == V:
            # Ensure it sums to 1 along the vocab dimension
            if not torch.allclose(x.sum(dim=2), torch.ones(B, T, device=x.device), atol=1e-4):
                raise ValueError("Input probabilities must sum to 1 along vocab dimension.")
            x = torch.matmul(x, self.embed.weight)  # [B, T, E]

        # Case 3: Precomputed embeddings [B, T, E]
        elif x.ndim == 3 and x.shape[2] == E:
            pass  # already in embedded space

        else:
            raise ValueError(f"Unrecognized input shape: {x.shape}")

        x = self.embed_dropout(x)
        initial_embeddings = x.clone()

        for layer in self.layers:
            x = layer(x)

        if self.normalize_before_projection:
            x = self.layer_norm(x)
        final_embeddings = x.clone()
        latent = self.latent_projection(x[:,0])  # project only the [CLS] token

        if return_internals:
            return latent, initial_embeddings, final_embeddings
        else:
            return latent

class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, sos_id, n_latents=8, embed_dropout=0.03):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.embed_dropout = nn.Dropout(embed_dropout)
        self.layers = nn.ModuleList([
            TransformerDecoderBlock(embed_dim, n_heads, ffn_dim, latent_dim, n_latents) for _ in range(n_layers)
        ])
        self.head = nn.Linear(embed_dim, vocab_size, bias=False)
        self.ln_f = nn.LayerNorm(embed_dim)
        self.context_window = context_window
        self.sos_id = sos_id

    def forward(self, x, z, return_internals=True):
        # x is the decoding seed, shape [B, T]
        B, T = x.shape

        # prepend <SOS> token
        x = torch.concat([torch.full((B, 1), self.sos_id, dtype=torch.long, device=x.device), x], dim=1)  # [B, T+1]

        x = self.embed(x)
        x = self.embed_dropout(x)

        for layer in self.layers:
            x = layer(x, z)

        x = self.ln_f(x)
        final_embeddings = x.clone()
        logits = self.head(x)
        if return_internals:
            return logits, final_embeddings
        else:
            return logits

class TransformerAutoencoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id, sos_id, n_latents=8, latent_dropout=0.03):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id)
        self.decoder = TransformerDecoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, sos_id, n_latents)
        self.latent_dropout = nn.Dropout(latent_dropout)

        self.cls_id = cls_id
        self.sos_id = sos_id
        self.context_window = context_window

        # weight tying of embeddings and head
        self.decoder.embed.weight = self.encoder.embed.weight
        self.decoder.head.weight = self.encoder.embed.weight

    def forward(self, x, decoding_seed, return_internals=True):
        B, T = x.shape
        latent, initial_embeddings, final_embeddings = self.encoder(x, return_internals=return_internals)
        logits, final_embeddings = self.decoder(decoding_seed, self.latent_dropout(latent), return_internals=return_internals)

        if return_internals:
            return logits, latent, initial_embeddings, final_embeddings
        else:
            return logits, latent

In [ ]:
# --------- 7. Training --------- WITH CHECKPOINT LOADING 26/06/25
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.epochs = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_losses = metrics_dict['train_losses']
        self.train_accuracies = metrics_dict['train_accuracies']
        self.val_losses = metrics_dict['val_losses']
        self.val_accuracies = metrics_dict['val_accuracies']
        self.epochs = list(range(1, len(self.train_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot losses
        ax1.plot(self.epochs, self.train_losses, label='Train Loss')
        ax1.plot(self.epochs, self.val_losses, label='Val Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.legend()
        ax1.grid(True)

        # Plot accuracies
        ax2.plot(self.epochs, self.train_accuracies, label='Train Accuracy')
        ax2.plot(self.epochs, self.val_accuracies, label='Val Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png")

        plt.show()

def load_checkpoint(checkpoint_path, model, device, optimizer=None):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch and loaded metrics
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    for key in ['vocab_size', 'embed_dim', 'n_layers', 'n_heads', 'ffn_dim', 'context_window']:
        if CONFIG[key] != saved_config[key]:
            raise ValueError(f"Checkpoint config mismatch for {key}: "
                           f"current={CONFIG[key]}, saved={saved_config[key]}")

    starting_epoch = checkpoint['epoch']
    return starting_epoch, checkpoint['metrics']

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if CONFIG['mode'] == 'LM':
        model = TinyLlamaTransformer(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window']
        ).to(CONFIG['device'])

    elif CONFIG['mode'] == 'AE':
        model = TransformerAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents']
        ).to(CONFIG['device'])

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, CONFIG['model_save_prefix'] + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    if CONFIG['checkpoint_path']:
        try:
            starting_epoch, saved_metrics = load_checkpoint(
                CONFIG['checkpoint_path'],
                model,
                CONFIG['device'],
                optimizer
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = CONFIG['lr']
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0

    print("Starting training loop...")
    global_start = time.time()

    def validate(model, val_loader, criterion, device):
        model.eval()
        total_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                if CONFIG['mode'] == 'LM':
                    logits = model(x)
                elif CONFIG['mode'] == 'AE':
                    logits, _, _, _ = model(x, decoding_seed=x[:, :-1])
                loss = criterion(logits.view(-1, CONFIG['vocab_size']), y.view(-1))
                acc = calculate_accuracy(logits, y, pad_id)
                total_loss += loss.item()
                total_acc += acc
        return total_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    for epoch in range(starting_epoch, CONFIG['epochs']):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(CONFIG['device'])
            y = y.to(CONFIG['device'])

            if CONFIG['mode'] == 'LM':
                logits = model(x)
            elif CONFIG['mode'] == 'AE':
                logits, _, _, _ = model(x, decoding_seed=x[:,:-1])
            loss = criterion(logits.view(-1, CONFIG['vocab_size']), y.view(-1))
            acc = calculate_accuracy(logits, y, pad_id)

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc

        # Calculate training metrics
        train_loss = epoch_loss / len(KAE_train_loader)
        train_acc = epoch_acc / len(KAE_train_loader)

        # Validation phase
        val_loss, val_acc = validate(model, val_loader, criterion, CONFIG['device'])

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_losses.append(train_loss)
        metrics.train_accuracies.append(train_acc)
        metrics.val_losses.append(val_loss)
        metrics.val_accuracies.append(val_acc)

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")

        # Plot metrics
        metrics.plot_metrics(save_dir)

        # Save checkpoint
        if (epoch+1) % CONFIG['save_every'] == 0 or (epoch+1) == CONFIG['epochs']:
            # ckpt_path = save_dir / f"{CONFIG['model_save_prefix']}_trial_{trial+1}_epoch{epoch+1}.pt"
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': CONFIG,
                'epoch': epoch+1,
                'metrics': {
                    'train_losses': metrics.train_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_losses': metrics.val_losses,
                    'val_accuracies': metrics.val_accuracies
                }
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

In [ ]:
import torch
import torch.nn.functional as F
from typing import List, Optional, Tuple

def generate_dyck(
    model: torch.nn.Module,
    prompt: Optional[List[str]] = None,
    max_new_tokens: int = 32,
    temperature: float = 1.0,
    top_k: int = 1,
    return_internals: bool = False,
    device: str = 'cuda'
) -> Tuple[List[str], torch.Tensor]:
    """
    Simple autoregressive generation function that returns both generated tokens and probabilities.

    Args:
        model: The trained transformer model
        prompt: Optional list of tokens to start generation with
        max_new_tokens: Maximum number of new tokens to generate
        temperature: Sampling temperature (1.0 = no change, < 1.0 = less random, > 1.0 = more random)
        top_k: Number of highest probability tokens to keep (0 = keep all)
        device: Device to run generation on ('cuda' or 'cpu')

    Returns:
        Tuple containing:
        - List of generated tokens
        - Tensor of shape [num_generated_tokens, vocab_size] containing probabilities for each step
    """
    # Setup model for inference
    model.eval()

    # BOS id
    bos_id = token2id['<BOS>']
    eos_id = token2id['<EOS>']

    # Initialize sequence with prompt or empty list if no prompt
    sequence = prompt.copy() if prompt else []

    # Get context window size from model config
    context_length = model.context_window

    # Convert sequence to tensor of token ids
    input_ids = torch.tensor(
        [bos_id] + [token2id[token] for token in sequence],
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    # Initialize list to store probabilities
    all_probs = []
    all_initial_embeds = []
    all_final_embeds = []

    # Generate tokens one at a time
    for _ in range(max_new_tokens):
        # If input is longer than context length, keep only the last context_length tokens
        if input_ids.size(1) > context_length:
            input_ids = input_ids[:, -context_length:]

        # Get model's output logits
        with torch.no_grad():
            if return_internals:
                logits, initial_embeddings, final_embeddings = model(input_ids, return_internals=True)
                logits = logits[:, -1, :]  # take logits for the last token
                all_initial_embeds.append(initial_embeddings.clone())
                all_final_embeds.append(final_embeddings.clone())
            else:
                logits = model(input_ids)[:, -1, :]

        # Apply temperature
        logits = logits / temperature
        logits_for_output = logits.clone()

        # Apply top-k filtering if specified
        if top_k > 0:
            top_k = min(top_k, logits.size(-1))
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = float('-inf')

        # Convert logits to probabilities
        probs = F.softmax(logits, dim=-1)

        # Store probabilities
        all_probs.append(F.softmax(logits_for_output, dim=-1)[0].clone())  # Add clone() to ensure we store a copy

        # Sample next token
        next_token_id = torch.multinomial(probs, num_samples=1)

        # Convert token id to token string
        next_token = id2token[next_token_id.item()]

        # Add to sequence
        sequence.append(next_token)

        # Update input_ids for next iteration
        input_ids = torch.cat([input_ids, next_token_id], dim=1)

        # Stop if EOS token is generated
        # if next_token == '<EOS>':
            # break

    # Stack all probabilities into a 2D tensor [num_steps, vocab_size]
    all_probs_tensor = torch.stack(all_probs, dim=0)

    if return_internals:
        return sequence, all_probs_tensor, all_initial_embeds, all_final_embeds
    else:
        return sequence, all_probs_tensor

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Optional
import numpy as np

def plot_generation_probabilities(
    probabilities: torch.Tensor,
    sequence: List[str],
    token2id: dict,
    id2token: dict,
    tokens_to_highlight: Optional[List[str]] = None,
    figsize: tuple = (12, 6),
    cmap: str = 'viridis',
    min_prob_threshold: float = 0.01
) -> None:
    """
    Plot token probabilities over generation steps.

    Args:
        probabilities: Tensor of shape [num_steps, vocab_size] containing probabilities
        sequence: List of generated tokens
        token2id: Dictionary mapping tokens to ids
        id2token: Dictionary mapping ids to tokens
        tokens_to_highlight: Optional list of specific tokens to highlight in the plot
        figsize: Figure size (width, height)
        cmap: Colormap to use
        min_prob_threshold: Minimum probability to show in the plot
    """
    # Convert probabilities to numpy
    probs_np = probabilities.cpu().numpy()
    num_steps, vocab_size = probs_np.shape

    # Create figure and axes
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=figsize, height_ratios=[3, 1])
    plt.subplots_adjust(hspace=0.3)

    # If no specific tokens are provided, show top-k at each step
    if tokens_to_highlight is None:
        tokens_to_highlight = ['(', ')', '[', ']', '<EOS>']

    # Get indices for tokens to highlight
    token_indices = [token2id[token] for token in tokens_to_highlight]

    # Plot probabilities for highlighted tokens
    for token, idx in zip(tokens_to_highlight, token_indices):
        probs = probs_np[:, idx]
        ax1.plot(range(num_steps), probs, label=token, marker='o', markersize=4)

    # Add chosen tokens markers
    for step, token in enumerate(sequence):
        token_id = token2id[token]
        prob = probs_np[step, token_id]
        ax1.scatter(step, prob, color='red', s=100, zorder=5,
                   marker='*', label='Chosen token' if step == 0 else "")

    # Customize upper plot
    ax1.set_title('Token Probabilities During Generation')
    ax1.set_xlabel('Generation Step')
    ax1.set_ylabel('Probability')
    ax1.grid(True, alpha=0.3)
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    # Add sequence visualization in lower plot
    token_colors = plt.cm.Set3(np.linspace(0, 1, len(tokens_to_highlight)))
    token_color_map = dict(zip(tokens_to_highlight, token_colors))

    # Create colored boxes for sequence
    for i, token in enumerate(sequence):
        color = token_color_map.get(token, 'gray')
        ax2.add_patch(plt.Rectangle((i, 0), 1, 1, facecolor=color))
        ax2.text(i + 0.5, 0.5, token, ha='center', va='center')

    # Customize lower plot
    ax2.set_xlim(0, len(sequence))
    ax2.set_ylim(0, 1)
    ax2.set_title('Generated Sequence')
    ax2.set_xticks(range(len(sequence)))
    ax2.set_yticks([])

    plt.tight_layout()
    return fig


In [ ]:
# Example generation
prompt = [token for token in '((']
return_internals = True
max_new_tokens = 100
temperature = 1
top_k = 1
if return_internals:
    generated, probs, input_embeds, output_embeds = generate_dyck(
        model=model,              # Your trained model
        prompt=prompt,            # Starting prompt
        max_new_tokens=max_new_tokens,       # Maximum new tokens to generate
        temperature=temperature,          # Lower = more focused
        top_k=top_k,
        return_internals=True,
        device=CONFIG['device']            # Run on GPU
    )
else:
    generated, probs = generate_dyck(
        model=model,              # Your trained model
        prompt=prompt,            # Starting prompt
        max_new_tokens=max_new_tokens,       # Maximum new tokens to generate
        temperature=temperature,          # Lower = more focused
        top_k=top_k,
        return_internals=False,
        device=CONFIG['device']            # Run on GPU
    )

# generated = decode(encode(generated))
print(''.join(generated))
print(f'Length: {len(generated)}')
print(f"Max Length: {len(generated) == len(prompt) + max_new_tokens}")
print(f'Valid: {is_valid_dyck2(generated, CONFIG["max_depth"])}')

# check if all sequences are valid
seqs = ''.join(generated).split('<EOS><BOS>')
for seq in seqs[:-1]:
  print(f'{seq} - {"Valid" if is_valid_dyck2(seq) else "Wrong"}')

# Create the visualization
fig = plot_generation_probabilities(
    probabilities=probs,
    sequence=generated[len(prompt):],
    token2id=token2id,
    id2token=id2token,
    tokens_to_highlight=None,
    figsize=(12, 8)
)
# print("Probability:")
# for id in range(probs.shape[-1]):
#   print(f"{id2token[id]}: {probs[0][id]:.5f}")

In [ ]:
# Loading a saved model
ckpt_path = CONFIG['checkpoint_path']
# ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
checkpoint = torch.load(ckpt_path, map_location=torch.device(CONFIG['device']))

# Load Tokenizer
# tokenizer_path = "tokenizer_tinyllama_tinystories_vocab_8k_bytelevel_24_06_25.json"
# tokenizer = Tokenizer.from_file(tokenizer_path)

if checkpoint['config']['mode'] == 'LM' or (not checkpoint['config']['mode']):
    model = TinyLlamaTransformer(
        vocab_size=checkpoint['config']['vocab_size'],
        embed_dim=checkpoint['config']['embed_dim'],
        n_layers=checkpoint['config']['n_layers'],
        n_heads=checkpoint['config']['n_heads'],
        ffn_dim=checkpoint['config']['ffn_dim'],
        context_window=checkpoint['config']['context_window']
    ).to(CONFIG['device'])

elif checkpoint['config']['mode'] == 'AE':
    model = TransformerAutoencoder(
            vocab_size=checkpoint['config']['vocab_size'],
            embed_dim=checkpoint['config']['embed_dim'],
            latent_dim=checkpoint['config']['latent_dim'],
            n_layers=checkpoint['config']['n_layers'],
            n_heads=checkpoint['config']['n_heads'],
            ffn_dim=checkpoint['config']['ffn_dim'],
            context_window=checkpoint['config']['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=checkpoint['config']['n_latents']
        ).to(CONFIG['device'])


model_state = checkpoint['model_state_dict']
model.load_state_dict(model_state)

In [ ]:
def get_mean_embedding(embedding_layer: nn.Embedding, vocab_probs: torch.Tensor) -> torch.Tensor:
    """
    Compute the mean embedding given an embedding layer and a probability distribution over the vocabulary.

    Args:
        embedding_layer (nn.Embedding): An embedding layer of shape (vocab_size, embedding_dim).
        vocab_probs (torch.Tensor): A 1D tensor of shape (vocab_size,) representing a probability distribution over the vocabulary.

    Returns:
        torch.Tensor: A tensor of shape (embedding_dim,) representing the mean embedding.
    """
    # Sanity check: make sure vocab_probs is normalized
    if not torch.isclose(vocab_probs.sum(), torch.tensor(1.0), atol=1e-4):
        raise ValueError("vocab_probs must sum to 1.")

    # Get the weight matrix from the embedding layer (shape: vocab_size x embedding_dim)
    embedding_weights = embedding_layer.weight  # shape: (vocab_size, embedding_dim)

    # Compute the mean embedding as a weighted sum
    mean_embedding = torch.matmul(vocab_probs, embedding_weights)  # shape: (embedding_dim,)

    return mean_embedding


In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def soft_generate(model, prompt_tokens, num_steps, device, mode="soft", tokenizer=None, temperature=1):
    """
    Soft autoregressive generation using expected embeddings instead of token sampling.
    Respects model's context window size.

    Args:
        model (TinyLlamaTransformer): The model instance.
        prompt_tokens (List or Tensor): if List: a list of length T0 of input tokens as strings. if Tensor: input embeddings of shape [B, T0, E].
        num_steps (int): Number of soft tokens to generate.
        device: device to use
        mode: "hard", "soft", or "raw"
        tokenizer: tokenizer for encoding the tokens
        temperature: Softmax temperature

    Returns:
        input_embeddings: [B, T0 + num_steps, E]
        output_embeddings: [B, num_steps, E]
        distributions: [B, num_steps, V]
        generated_tokens: [B, num_steps]  # argmax token IDs at each step
    """

    # Setup model for inference
    model.eval()

    # BOS id
    if tokenizer is None:
      bos_id = token2id['<BOS>']
      eos_id = token2id['<EOS>']
    else:
      bos_id = tokenizer.token_to_id("[BOS]")
      eos_id = tokenizer.token_to_id("[EOS]")


    # expected dimensions
    E = model.embed.embedding_dim
    V = model.embed.num_embeddings
    context_window = model.context_window

    if isinstance(prompt_tokens, List):
        # Initialize sequence with prompt or empty list if no prompt
        sequence = prompt_tokens.copy() if prompt_tokens else []

        # Convert sequence to tensor of token ids
        if tokenizer is None:
            input_ids = torch.tensor(
                [bos_id] + [token2id[token] for token in sequence],
                dtype=torch.long,
                device=device
            ).unsqueeze(0)
        else:
            input_ids = torch.tensor(
                [bos_id] + tokenizer.encode(''.join(sequence)).ids,
                dtype=torch.long,
                device=device
            ).unsqueeze(0)

        # Tokens → Embeddings
        input_embeds = model.embed(input_ids)  # [B, T0, E]
        B, T0 = input_ids.shape

    elif isinstance(prompt_tokens, torch.Tensor):
        # Raw embeddings
        sequence = []
        input_embeds = prompt_tokens.to(device)
        B, T0, E_check = input_embeds.shape
        assert E_check == E, f"Expected embedding dim {E}, got {E_check}"
    else:
        raise ValueError("prompt_tokens must be either token IDs (long) or embeddings (float)")

    # Embed initial prompt
    all_input_embeds = [input_embeds]
    output_embeds = []
    distributions = []
    generated_tokens = sequence

    for step in range(num_steps):
        # Truncate input to context window
        current_input = torch.cat(all_input_embeds, dim=1)
        if current_input.size(1) > context_window:
            current_input = current_input[:, -context_window:, :]

        # Forward pass (using embedded input)
        logits, _, final_embeds = model(current_input, return_internals=True)
        last_logits = logits[:, -1, :] / temperature  # [B, V]
        probs = F.softmax(last_logits, dim=-1)        # [B, V]
        last_out_embed = final_embeds[:, -1, :]       # [B, E]

        # Choose next input embedding based on mode
        if mode == "raw":  # raw embeddings
            next_input_embed = last_out_embed       # [B, E]
        elif mode == "soft": # re-embed using probabilities
            next_input_embed = torch.matmul(probs, model.embed.weight)  # [B, E]
        elif mode == "hard": # regular decoding
            next_input_embed = model.embed(probs.argmax(dim=-1))  # [B, E]
        else:
            raise ValueError(f"Unknown mode: {mode}")

        # Record data
        output_embeds.append(last_out_embed)
        distributions.append(probs)
        if tokenizer is None:
            generated_tokens.append(id2token[probs.argmax(dim=-1).item()])  # [B]
        else:
            generated_tokens.append(tokenizer.decode([probs.argmax(dim=-1).item()], skip_special_tokens=False))  # [B]
        all_input_embeds.append(next_input_embed.unsqueeze(1))  # [B, 1, E]

    # Stack and return
    input_embeddings = torch.cat(all_input_embeds, dim=1)            # [B, T0 + num_steps, E]
    output_embeddings = torch.stack(output_embeds, dim=1)            # [B, num_steps, E]
    distributions = torch.stack(distributions, dim=1)                # [B, num_steps, V]
    # generated_tokens = torch.stack(generated_tokens, dim=1)          # [B, num_steps]

    return input_embeddings, output_embeddings, distributions, generated_tokens


In [ ]:
import torch
import torch.nn.functional as F
from typing import List, Optional, Union

@torch.no_grad()
def soft_generate_batch(
    model,
    prompt_tokens: Union[List[List[str]], torch.Tensor],
    num_steps: int,
    device,
    mode="soft",
    tokenizer=None,
    temperature=1
):
    """
    Batched soft autoregressive generation using expected embeddings instead of token sampling.
    Handles batch input for both token lists and embeddings.

    Args:
        model (TinyLlamaTransformer): The model instance.
        prompt_tokens (List[List[str]] or Tensor): 
            - List[List[str]]: batch of token sequences (strings).
            - Tensor: input embeddings of shape [B, T0, E].
        num_steps (int): Number of soft tokens to generate.
        device: device to use.
        mode: "hard", "soft", or "raw".
        tokenizer: tokenizer for encoding the tokens.
        temperature: Softmax temperature.

    Returns:
        input_embeddings: [B, T0 + num_steps, E]
        output_embeddings: [B, num_steps, E]
        distributions: [B, num_steps, V]
        generated_tokens: List[List[str]]  # argmax token IDs at each step for each batch
    """
    model.eval()

    # BOS/EOS ids
    if tokenizer is None:
        bos_id = token2id['<BOS>']
        eos_id = token2id['<EOS>']
    else:
        bos_id = tokenizer.token_to_id("[BOS]")
        eos_id = tokenizer.token_to_id("[EOS]")

    E = model.embed.embedding_dim
    V = model.embed.num_embeddings
    context_window = model.context_window

    # Handle prompt_tokens as batch of token lists
    if isinstance(prompt_tokens, list) and isinstance(prompt_tokens[0], list):
        batch_size = len(prompt_tokens)
        # Convert each sequence to ids
        if tokenizer is None:
            input_ids = [
                [bos_id] + [token2id[token] for token in seq]
                for seq in prompt_tokens
            ]
        else:
            input_ids = [
                [bos_id] + tokenizer.encode(''.join(seq)).ids
                for seq in prompt_tokens
            ]
        # Pad to same length
        max_len = max(len(ids) for ids in input_ids)
        input_ids = [
            ids + [eos_id] * (max_len - len(ids)) for ids in input_ids
        ]
        input_ids = torch.tensor(input_ids, dtype=torch.long, device=device)  # [B, T0]
        input_embeds = model.embed(input_ids)  # [B, T0, E]
        B, T0 = input_ids.shape
        generated_tokens = [seq.copy() for seq in prompt_tokens]
    elif isinstance(prompt_tokens, torch.Tensor):
        input_embeds = prompt_tokens.to(device)
        B, T0, E_check = input_embeds.shape
        assert E_check == E, f"Expected embedding dim {E}, got {E_check}"
        generated_tokens = [[] for _ in range(B)]
    else:
        raise ValueError("prompt_tokens must be List[List[str]] or Tensor [B, T0, E]")

    all_input_embeds = [input_embeds]
    output_embeds = []
    distributions = []

    for step in range(num_steps):
        current_input = torch.cat(all_input_embeds, dim=1)
        if current_input.size(1) > context_window:
            current_input = current_input[:, -context_window:, :]

        logits, _, final_embeds = model(current_input, return_internals=True)
        last_logits = logits[:, -1, :] / temperature  # [B, V]
        probs = F.softmax(last_logits, dim=-1)        # [B, V]
        last_out_embed = final_embeds[:, -1, :]       # [B, E]

        # Choose next input embedding based on mode
        if mode == "raw":
            next_input_embed = last_out_embed
        elif mode == "soft":
            next_input_embed = torch.matmul(probs, model.embed.weight)  # [B, E]
        elif mode == "hard":
            next_input_embed = model.embed(probs.argmax(dim=-1))  # [B, E]
        else:
            raise ValueError(f"Unknown mode: {mode}")

        # Record data
        output_embeds.append(last_out_embed)
        distributions.append(probs)
        # Update generated tokens for each batch
        for i in range(B):
            token_id = probs[i].argmax().item()
            if tokenizer is None:
                generated_tokens[i].append(id2token[token_id])
            else:
                generated_tokens[i].append(tokenizer.decode([token_id], skip_special_tokens=False))
        all_input_embeds.append(next_input_embed.unsqueeze(1))  # [B, 1, E]

    input_embeddings = torch.cat(all_input_embeds, dim=1)            # [B, T0 + num_steps, E]
    output_embeddings = torch.stack(output_embeds, dim=1)            # [B, num_steps, E]
    distributions = torch.stack(distributions, dim=1)                # [B, num_steps, V]

    return input_embeddings,output_embeddings, distributions, generated_tokens

In [ ]:
def stack_context_windows(output_embeddings: torch.Tensor, context_window: int, pad: bool = False) -> torch.Tensor:
    """
    Stack overlapping context windows from output_embeddings, with optional zero padding at the start.

    Args:
        output_embeddings (Tensor): [B, T, E] — output embedding sequences
        context_window (int): number of consecutive steps to stack
        pad (bool): If True, pad the first context_window - 1 steps with zeros so output has same length T.
                    If False, ignore the first context_window - 1 steps, resulting in output length T - context_window + 1.

    Returns:
        Tensor: [B, T', E * context_window] — stacked windowed embeddings
                where T' = T if pad=True, else T - context_window + 1
    """
    B, T, E = output_embeddings.shape

    if pad:
        # Pad with zeros at the beginning: [B, context_window - 1, E]
        padding = torch.zeros(B, context_window - 1, E, device=output_embeddings.device, dtype=output_embeddings.dtype)
        padded_embeddings = torch.cat([padding, output_embeddings], dim=1)  # shape: [B, T + context_window - 1, E]
        T_new = T
    else:
        if T < context_window:
            raise ValueError(f"Input sequence length T={T} is smaller than context_window={context_window}.")
        padded_embeddings = output_embeddings
        T_new = T - context_window + 1

    windows = []
    for t in range(T_new):
        # shape: [B, context_window, E]
        window = padded_embeddings[:, t:t + context_window, :]
        # shape: [B, E * context_window]
        window_flat = window.reshape(B, -1)
        windows.append(window_flat)

    # shape: [B, T', E * context_window]
    return torch.stack(windows, dim=1)


In [ ]:
def stack_context_windows_tokens(trajectory_tokens, context_window: int, pad: bool = False):
    """
    Stack overlapping context windows from output_embeddings, with optional zero padding at the start.

    Args:
        output_embeddings (Tensor): [B, T, E] — output embedding sequences
        context_window (int): number of consecutive steps to stack
        pad (bool): If True, pad the first context_window - 1 steps with zeros so output has same length T.
                    If False, ignore the first context_window - 1 steps, resulting in output length T - context_window + 1.

    Returns:
        Tensor: [B, T', E * context_window] — stacked windowed embeddings
                where T' = T if pad=True, else T - context_window + 1
    """
    B, T = trajectory_tokens.shape

    if pad:
        # Pad with zeros at the beginning: [B, context_window - 1, E]
        padding = np.zeros(B, context_window - 1)
        padded_embeddings = np.cat([padding, trajectory_tokens], dim=1)  # shape: [B, T + context_window - 1, E]
        T_new = T
    else:
        if T < context_window:
            raise ValueError(f"Input sequence length T={T} is smaller than context_window={context_window}.")
        padded_embeddings = trajectory_tokens
        T_new = T - context_window + 1

    windows = []
    for t in range(T_new):
        # shape: [B, context_window]
        window = padded_embeddings[:, t:t + context_window]
        windows.append(window)

    # shape: [B, T', context_window]
    return np.stack(windows, axis=1)


In [ ]:
import matplotlib.pyplot as plt
import torch

def plot_token_distributions(distributions, vocab=None, title="Token Probability Surface"):
    """
    Plots a 2D surface (heatmap) of token probabilities across time steps.

    Args:
        distributions (Tensor): [B, T, V] tensor of probabilities.
        vocab (list of str, optional): Token strings for y-axis labels.
        title (str): Plot title.
    """
    # Use only the first batch item
    dist = distributions[0].cpu().detach().numpy()  # shape [T, V]
    dist = dist.T  # shape [V, T] so tokens on y-axis, time on x-axis

    plt.figure(figsize=(12, 6))
    im = plt.imshow(dist, aspect='auto', interpolation='nearest', origin='lower', cmap='viridis')

    plt.colorbar(im, label="Probability")
    plt.xlabel("Time step")
    plt.ylabel("Token index" if vocab is None else "Token")
    plt.title(title)

    if vocab is not None and len(vocab) <= 100:  # avoid overcrowding
        plt.yticks(ticks=range(len(vocab)), labels=vocab)
    elif vocab is not None:
        print("Too many vocab items to label y-axis.")

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap

# Get the built-in colormaps
reds = cm.get_cmap('Reds')
greens = cm.get_cmap('Greens')
blues = cm.get_cmap('Blues')

# Define new colormaps that sample only the upper half (0.5 to 1.0)
reds_half = LinearSegmentedColormap.from_list('Reds_half', reds(np.linspace(0.33, 1.0, 256)))
greens_half = LinearSegmentedColormap.from_list('Greens_half', greens(np.linspace(0.33, 1.0, 256)))
blues_half = LinearSegmentedColormap.from_list('Blues_half', blues(np.linspace(0.33, 1.0, 256)))

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection
import numpy as np

def plot_output_embeddings_pca_3d(output_embeddings, title="3D PCA of Output Embeddings", pca=None, add_line=False):
    """
    Plots a 3D scatter plot of the PCA-reduced output embeddings over time.

    Args:
        output_embeddings (Tensor or list of Tensors): each tensor [B, T, E]
        title (str): plot title
        pca (PCA): an optional PCA object to use for dimensionality reduction instead of fitting a new one
    """
    # If it's not a list, make it a list of one element
    if not isinstance(output_embeddings, list):
        output_embeddings = [output_embeddings]

    # Check all shapes
    B, T, E = output_embeddings[0].shape
    for tensor in output_embeddings:
        assert tensor.shape == (B, T, E), "All tensors must have the same shape"

    # Concatenate along time dimension
    embeddings_cat = torch.cat(output_embeddings, dim=1)  # [B, T*num, E]
    B, T_total, E = embeddings_cat.shape
    embeddings_flat = embeddings_cat.cpu().detach().reshape(B * T_total, E)

    # Fit or use PCA
    if pca is None:
        pca = PCA(n_components=3)
        embeddings_3d_cat = pca.fit_transform(embeddings_flat)  # [B*T_total, 3]
    else:
        embeddings_3d_cat = pca.transform(embeddings_flat)

    # Calculate explained variance ratio
    explained_variance = pca.explained_variance_ratio_
    explained_variance_str = f"Explained Variance: PC1={explained_variance[0]:.2f}, PC2={explained_variance[1]:.2f}, PC3={explained_variance[2]:.2f}"

    # Transform each original tensor individually
    embeddings_3d_list = []
    for tensor in output_embeddings:
        embeddings_flat = tensor.cpu().detach().reshape(B * T, E)
        embeddings_3d = pca.transform(embeddings_flat)
        embeddings_3d_list.append(embeddings_3d)

    # Colormaps for each embedding set
    colormaps = [reds_half, greens_half, blues_half, 'Reds', 'Greens', 'Blues', 'plasma', 'cool', 'spring', 'magma', 'viridis', 'cividis', 'inferno', 'winter']
    if len(embeddings_3d_list) > len(colormaps):
        raise ValueError("Not enough predefined colormaps for number of embedding sets!")

    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')

    for i, embeddings_3d in enumerate(embeddings_3d_list):
        time_steps = torch.arange(T).repeat(B).numpy()
        sc = ax.scatter(
            embeddings_3d[:, 0],
            embeddings_3d[:, 1],
            embeddings_3d[:, 2],
            c=time_steps,
            cmap=colormaps[i],
            s=30,
            alpha=0.8,
            label=f'Set {i+1}'
        )
        if add_line:
            ax.plot(embeddings_3d[:, 0],
            embeddings_3d[:, 1],
            embeddings_3d[:, 2],
            linewidth=0.5),

        cbar = plt.colorbar(sc, pad=0.05)
        cbar.set_label(f"Time Step (Set {i+1})")

    ax.set_title(f"{title}\n{explained_variance_str}")
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.set_zlabel("PCA 3")
    ax.legend()
    plt.tight_layout()
    plt.show()

    return pca


In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D  # needed for 3D projection

def plot_output_embeddings_3d(output_embeddings, title="Output Embeddings"):
    """
    Plots a 3D scatter plot of the output embeddings over time.

    Args:
        output_embeddings (Tensor): [B, T, E] tensor
        title (str): plot title
    """
    B, T, E = output_embeddings.shape
    embeddings_3d = output_embeddings.cpu().detach().reshape(B * T, E)  # [B*T, E]

    # Create color mapping by time
    time_steps = torch.arange(T).repeat(B).numpy()  # [B*T]

    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(
        embeddings_3d[:, 0],
        embeddings_3d[:, 1],
        embeddings_3d[:, 2],
        c=time_steps,
        cmap='plasma',
        s=30,
        alpha=0.9
    )

    ax.set_title(title)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    cbar = plt.colorbar(sc, pad=0.1)
    cbar.set_label("Time Step")
    plt.tight_layout()
    plt.show()

    return


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.animation import FuncAnimation
import matplotlib as mpl
from IPython.display import HTML

def animate_pca_trajectory(output_embeddings, save_path=None, interval=100, title="PCA Trajectory Animation", pca=None):
    """
    Creates a 3D animation of PCA-reduced output_embeddings over time.

    Args:
        output_embeddings (Tensor): [B, T, E] tensor (only B=1 supported for animation)
        save_path (str or None): Path to save the animation (e.g., "trajectory.mp4"). If None, shows inline.
        interval (int): Delay between frames in milliseconds.
        title (str): Title for the plot.
    """
    B, T, E = output_embeddings.shape
    if B != 1:
        raise ValueError("This animation function currently supports only batch size B=1.")

    embeddings = output_embeddings[0].cpu().detach().numpy()  # [T, E]

    # Reduce to 3D using PCA
    if pca is None:
        pca = PCA(n_components=3)
        embeddings_3d = pca.fit_transform(embeddings)  # [B*T, 3]
    else:
        embeddings_3d = pca.transform(embeddings)  # [B*T, 3]

    # Setup plot
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    ax.set_xlim(np.min(embeddings_3d[:, 0]), np.max(embeddings_3d[:, 0]))
    ax.set_ylim(np.min(embeddings_3d[:, 1]), np.max(embeddings_3d[:, 1]))
    ax.set_zlim(np.min(embeddings_3d[:, 2]), np.max(embeddings_3d[:, 2]))
    ax.set_title(title)
    ax.set_xlabel("PCA 1")
    ax.set_ylabel("PCA 2")
    ax.set_zlabel("PCA 3")

    # Line and point elements
    line, = ax.plot([], [], [], color='blue', linewidth=0.5)
    point, = ax.plot([], [], [], 'ro')

    def init():
        line.set_data([], [])
        line.set_3d_properties([])
        point.set_data([], [])
        point.set_3d_properties([])
        return line, point

    def update(frame):
        line.set_data(embeddings_3d[:frame+1, 0], embeddings_3d[:frame+1, 1])
        line.set_3d_properties(embeddings_3d[:frame+1, 2])
        point.set_data(embeddings_3d[frame, 0:1], embeddings_3d[frame, 1:2])
        point.set_3d_properties(embeddings_3d[frame, 2:3])
        return line, point

    anim = FuncAnimation(fig, update, frames=T, init_func=init, interval=interval, blit=True)

    if save_path:
        anim.save(save_path, fps=1000 // interval, dpi=200)
    else:
        # plt.show()
        pass

    mpl.rcParams['animation.embed_limit'] = 150  # value in MB, increase as needed
    return HTML(anim.to_jshtml())


In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mpl_toolkits.mplot3d import Axes3D  # registers 3D projection

def plot_output_embeddings_tsne_3d(output_embeddings, title="3D t-SNE of Output Embeddings", perplexity=30, random_seed=42):
    """
    Plots a 3D t-SNE scatter of the output embeddings over time.

    Args:
        output_embeddings (Tensor): [B, T, E] tensor
        title (str): plot title
        perplexity (float): t-SNE perplexity (recommend < T)
        random_seed (int): for reproducibility
    """
    B, T, E = output_embeddings.shape
    embeddings = output_embeddings.cpu().detach().reshape(B * T, E).numpy()  # [B*T, E]

    # Time steps for color
    time_steps = torch.arange(T).repeat(B).numpy()  # [B*T]

    # Run t-SNE
    tsne = TSNE(n_components=3, perplexity=perplexity, random_state=random_seed, init='pca')
    embeddings_3d = tsne.fit_transform(embeddings)  # [B*T, 3]

    # Plotting
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')
    sc = ax.scatter(
        embeddings_3d[:, 0],
        embeddings_3d[:, 1],
        embeddings_3d[:, 2],
        c=time_steps,
        cmap='plasma',
        s=30,
        alpha=0.9
    )

    ax.set_title(title)
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.set_zlabel("t-SNE 3")
    cbar = plt.colorbar(sc, pad=0.1)
    cbar.set_label("Time Step")
    plt.tight_layout()
    plt.show()


In [ ]:
def soft_generate_all_modes(model, prompt, max_new_tokens, device, tokenizer=None, temperature=1):
    input_embeddings_hard, output_embeddings_hard, distributions_hard, generated_hard = soft_generate(model,
                                                                              prompt,
                                                                              max_new_tokens,
                                                                              device=CONFIG['device'],
                                                                              mode="hard",
                                                                              tokenizer=tokenizer,
                                                                              temperature=temperature)

    input_embeddings_soft, output_embeddings_soft, distributions_soft, generated_soft = soft_generate(model,
                                                                              prompt,
                                                                              max_new_tokens,
                                                                              device=CONFIG['device'],
                                                                              mode="soft",
                                                                              tokenizer=tokenizer,
                                                                              temperature=temperature)

    input_embeddings_raw, output_embeddings_raw, distributions_raw, generated_raw = soft_generate(model,
                                                                              prompt,
                                                                              max_new_tokens,
                                                                              device=CONFIG['device'],
                                                                              mode="raw",
                                                                              tokenizer=tokenizer,
                                                                              temperature=temperature)

    return input_embeddings_hard, output_embeddings_hard, distributions_hard, generated_hard,  \
    input_embeddings_soft, output_embeddings_soft, distributions_soft, generated_soft,  \
    input_embeddings_raw, output_embeddings_raw, distributions_raw, generated_raw



In [ ]:
def pearson_cross_correlation(tensor1, tensor2, center=True):
    if center:
        tensor1_c = tensor1 - tensor1.mean(dim=1, keepdim=True)
        tensor2_c = tensor2 - tensor2.mean(dim=1, keepdim=True)
    else:
        tensor1_c = tensor1
        tensor2_c = tensor2

    dot = tensor1_c @ tensor2_c.T  # [T, T]

    tensor1_norm = tensor1_c.norm(dim=1)
    tensor2_norm = tensor2_c.norm(dim=1)

    corr = dot / torch.outer(tensor1_norm, tensor2_norm)  # [T, T]

    return corr


In [ ]:
# Example soft generation
# prompt = [token for token in '(([']
# prompt = [token for token in '(())[][]([[]])(())()()()[]([()])']
# prompt = [token for token in '(([()[']
prompt = [token for token in '(()[][][[]])([()()])(()[][][[]])([()()])']
# prompt = [token for token in '(()[][][[]])([()()])(()[][][[']
# prompt = ['[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>']
# prompt = torch.randn(1, model.context_window, model.embed.embedding_dim) * 100
# prompt = torch.ones(1, model.context_window, model.embed.embedding_dim) * 1
# prompt = -1 * output_embeddings[:, -1].unsqueeze(1).repeat(1,model.context_window,1)
# prompt = [token for token in """Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, she saw a big, scary dog. The dog was barking and running around. Lily was scared and ran away.

# Lily's mom came to her room and said, "Don't worry, Lily. I will protect you." She took Lily to the park and they played together. They ran and ran, but the dog was too fast. Lily was sad and cried.

# Then, Lily's mom came and saw the dog. She said, "Don't worry, Lily. I will protect you." Lily felt better and they went home. The dog was happy and wagged his tail. Lily was happy too, and she knew that she would protect her mom and her dog."""]
# prompt = [token for token in 'Once upon a time']
max_new_tokens = 10000
temperature = 0.04
mode = "soft"  # hard / soft / raw

input_embeddings, trajectory_tokens, distributions, generated = soft_generate(model,
                                                                              prompt,
                                                                              max_new_tokens,
                                                                              device=CONFIG['device'],
                                                                              mode=mode,
                                                                              tokenizer=None,
                                                                              temperature=temperature)


print(''.join(generated))
print("")
print(f'Length: {len(generated)}')
print(f"Max Length: {len(generated) == len(prompt) + max_new_tokens}")
print(f'Valid: {is_valid_dyck2(generated, CONFIG["max_depth"])}')


In [ ]:
# soft generation - all modes
prompt = [token for token in '[][]()[()(()()[[()(())]([][()()]()())()])]']
# prompt = [token for token in '[][][]()()[]()[][][][][][][][]][][][]()()[]()[][][][][][][][][][][]()()[]()[][][][][][][][]][][][]()()[]()[][][][][][][][]']
# prompt = [token for token in '(([()[']
# prompt = [token for token in '(()[][][[]])([()()])(()[][][[]])([()()])']
# prompt = [token for token in '(()[][][[]])([()()])(()[][][[']
# prompt = ['[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>','[',']','(',')','[',']','(',')','(',')','[',']','<EOS>','<BOS>']
# prompt = ['(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')',')','<EOS>','<BOS>','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')',')','<EOS>','<BOS>']
# prompt = ['(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','<EOS>','<BOS>','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','(',')','<EOS>','<BOS>']
# prompt = torch.randn(1, model.context_window, model.embed.embedding_dim) * 100
# prompt = torch.ones(1, model.context_window, model.embed.embedding_dim) * 1
# prompt = -1 * output_embeddings[:, 199].unsqueeze(1).repeat(1,model.context_window,1)
# prompt = [token for token in """Once upon a time, there was a little girl named Lily. She loved to play outside in the park. One day, she saw a big, scary dog. The dog was barking and running around. Lily was scared and ran away.

# Lily's mom came to her room and said, "Don't worry, Lily. I will protect you." She took Lily to the park and they played together. They ran and ran, but the dog was too fast. Lily was sad and cried.

# Then, Lily's mom came and saw the dog. She said, "Don't worry, Lily. I will protect you." Lily felt better and they went home. The dog was happy and wagged his tail. Lily was happy too, and she knew that she would protect her mom and her dog."""]
# prompt = [token for token in 'It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash. It does not like the ash.']
max_new_tokens = 1000
temperature = 0.00001

input_embeddings_hard, output_embeddings_hard, distributions_hard, generated_hard,  \
input_embeddings_soft, output_embeddings_soft, distributions_soft, generated_soft,  \
input_embeddings_raw, output_embeddings_raw, distributions_raw, generated_raw = soft_generate_all_modes(model,
                                                                              prompt,
                                                                              max_new_tokens,
                                                                              device=CONFIG['device'],
                                                                              tokenizer=None,
                                                                              temperature=temperature)


print(''.join(generated_hard))
# print(''.join(generated_soft))
# print("")
# print(f'Length: {len(generated)}')
# print(f"Max Length: {len(generated) == len(prompt) + max_new_tokens}")
# print(f'Valid: {is_valid_dyck2(generated, CONFIG["max_depth"])}')


In [ ]:
# Soft Generation Plots

pca = plot_output_embeddings_pca_3d(trajectory_tokens[:, model.context_window:], title="3D PCA of Output Embeddings")
plot_output_embeddings_pca_3d(input_embeddings, pca=None, title="3D PCA of Input Embeddings")
# plot_output_embeddings_tsne_3d(output_embeddings)

plot_token_distributions(distributions)
plot_output_embeddings_3d(trajectory_tokens[:, model.context_window:])

In [ ]:
stacked_input_embeddings = stack_context_windows(input_embeddings, context_window=model.context_window, pad=False)
stacked_output_embeddings = stack_context_windows(trajectory_tokens, context_window=model.context_window, pad=False)
pca = plot_output_embeddings_pca_3d(stacked_output_embeddings, title="3D PCA of Stacked Output Embeddings")
# plot_output_embeddings_pca_3d(stacked_output_embeddings, title="3D PCA of Stacked Output Embeddings")
plot_output_embeddings_pca_3d(stacked_input_embeddings, pca=None, title="3D PCA of Stacked Input Embeddings")

stacked_embeddings_padded = stack_context_windows(input_embeddings, context_window=model.context_window, pad=True)
# plot_output_embeddings_pca_3d(stacked_embeddings_padded, pca=pca, title="3D PCA of Stacked Input Embeddings Padded")

# plot_output_embeddings_tsne_3d(stacked_output_embeddings, title="3D t-SNE of Stacked Output Embeddings")
# plot_output_embeddings_tsne_3d(stacked_input_embeddings, title="3D t-SNE of Stacked Input Embeddings")

In [ ]:
stacked_input_embeddings_hard = stack_context_windows(input_embeddings_hard[:,:], context_window=model.context_window, pad=False)
stacked_input_embeddings_soft = stack_context_windows(input_embeddings_soft[:,:], context_window=model.context_window, pad=False)
stacked_input_embeddings_raw = stack_context_windows(input_embeddings_raw[:,:], context_window=model.context_window, pad=False)

stacked_output_embeddings_hard = stack_context_windows(output_embeddings_hard, context_window=model.context_window, pad=False)
stacked_output_embeddings_soft = stack_context_windows(output_embeddings_soft, context_window=model.context_window, pad=False)
stacked_output_embeddings_raw = stack_context_windows(output_embeddings_raw, context_window=model.context_window, pad=False)

# Output Embeddings
stacked_concat_output_embeddings = torch.cat([stacked_output_embeddings_hard, stacked_output_embeddings_soft, stacked_output_embeddings_raw], dim=1)
pca = plot_output_embeddings_pca_3d([stacked_output_embeddings_hard, stacked_output_embeddings_soft, stacked_output_embeddings_raw], pca=None, title="3D PCA of Stacked Concat Output Embeddings")
pca_outputs = plot_output_embeddings_pca_3d([stacked_output_embeddings_hard, stacked_output_embeddings_soft], pca=None, title="3D PCA of Stacked Hard+Soft Output Embeddings")

# Input Embeddings
offset = 500
stacked_concat_input_embeddings = torch.cat([stacked_input_embeddings_hard, stacked_input_embeddings_soft, stacked_input_embeddings_raw], dim=1)
pca = plot_output_embeddings_pca_3d([stacked_input_embeddings_hard, stacked_input_embeddings_soft, stacked_input_embeddings_raw], pca=None, title="3D PCA of Stacked Concat Input Embeddings")
pca_inputs = plot_output_embeddings_pca_3d([stacked_input_embeddings_hard, stacked_input_embeddings_soft], pca=None, title="3D PCA of Stacked Hard+Soft Input Embeddings")
plot_output_embeddings_pca_3d([stacked_input_embeddings_hard[:, offset:], stacked_input_embeddings_soft[:, offset:]], pca=None, title="3D PCA of Stacked Hard+Soft Input Embeddings with Offset")
plot_output_embeddings_pca_3d([stacked_input_embeddings_raw[:, offset:]], pca=None, title="3D PCA of Stacked Raw Input Embeddings")
plot_output_embeddings_pca_3d([stacked_input_embeddings_hard[:, offset:]], pca=None, title="3D PCA of Stacked Hard Input Embeddings")
plot_output_embeddings_pca_3d([stacked_input_embeddings_soft[:, offset:]], pca=None, title="3D PCA of Stacked Soft Input Embeddings", add_line=True)

# plot_output_embeddings_tsne_3d(stacked_concat_output_embeddings, title="3D t-SNE of Concatanated Output Embeddings")
# plot_output_embeddings_tsne_3d(stacked_concat_input_embeddings, title="3D t-SNE of Concatanated Input Embeddings")
# plot_output_embeddings_tsne_3d(stacked_output_embeddings_soft, title="3D t-SNE of Soft Output Embeddings")

In [ ]:
# pca_outputs = plot_output_embeddings_pca_3d([stacked_output_embeddings_soft[:,530:570]], pca=pca_outputs, title="3D PCA of Stacked Hard+Soft Output Embeddings", add_line=True)
# plot_token_distributions(distributions_hard[:,400:800])
plot_token_distributions(distributions_soft[:,1600:2000])
# plot_token_distributions(distributions_soft[:,400:800] - distributions_hard[:,400:800])

In [ ]:
center_offset = 400
display_offset = 400

# corr = pearson_cross_correlation(stacked_input_embeddings_hard.squeeze(), stacked_input_embeddings_raw.squeeze())
# corr = pearson_cross_correlation(stacked_output_embeddings_hard.squeeze(), stacked_output_embeddings_raw.squeeze())
# corr = pearson_cross_correlation(stacked_input_embeddings_hard[:, 100:].squeeze().T, stacked_input_embeddings_raw[:, 100:].squeeze().T)
# points1_c = points1 - points1.mean(dim=0, keepdim=True)
# points2_c = points2 - points2.mean(dim=0, keepdim=True)
stacked_input_embeddings_hard_c = stacked_input_embeddings_hard - stacked_input_embeddings_hard[:,center_offset:].mean(dim=1, keepdim=True)
stacked_input_embeddings_raw_c = stacked_input_embeddings_raw - stacked_input_embeddings_raw[:,center_offset:].mean(dim=1, keepdim=True)
stacked_input_embeddings_soft_c = stacked_input_embeddings_soft - stacked_input_embeddings_soft[:,center_offset:].mean(dim=1, keepdim=True)
stacked_input_embeddings_c = stacked_input_embeddings - stacked_input_embeddings[:,center_offset:].mean(dim=1, keepdim=True)

center = True
# corr = pearson_cross_correlation(stacked_input_embeddings_hard.squeeze(), stacked_input_embeddings_raw.squeeze(), center=center)
# corr = pearson_cross_correlation(stacked_input_embeddings_hard_c.squeeze(), stacked_input_embeddings_raw_c.squeeze(), center=center)
# corr = pearson_cross_correlation(stacked_input_embeddings_raw_c.squeeze(), stacked_input_embeddings_raw_c.squeeze(), center=center)
# corr = pearson_cross_correlation(stacked_input_embeddings_hard_c.squeeze(), stacked_input_embeddings_hard_c.squeeze(), center=center)
corr = pearson_cross_correlation(stacked_input_embeddings_soft_c.squeeze(), stacked_input_embeddings_soft_c.squeeze(), center=center)
# corr = pearson_cross_correlation(stacked_input_embeddings_c.squeeze(), stacked_input_embeddings_c.squeeze(), center=center)
# corr = pearson_cross_correlation(points1, points2)

corr_np = corr.cpu().numpy()
# corr_np = 5 * np.clip(corr_np, -0.2, 0.2)
# corr_np = corr_np - corr_np.mean(axis=0, keepdims=True)
# corr_np = corr_np - corr_np.mean(axis=1, keepdims=True)

# corr_np = corr_np[display_offset:, display_offset:]
# corr_np = corr_np - corr_np.mean(axis=0, keepdims=True)
# corr_np = corr_np - corr_np.mean(axis=1, keepdims=True)

# 7. Plot heatmap
plt.figure(figsize=(15, 10))
# plt.imshow((corr_np), cmap='coolwarm', vmin=-1, vmax=1)
max_abs = np.abs(corr_np).max()
plt.imshow((corr_np[:]), cmap='coolwarm', vmin=-max_abs, vmax=max_abs)
# plt.imshow((corr_np[:]), cmap='coolwarm', vmin=corr_np.min(), vmax=corr_np.max())
plt.colorbar(label='Pearson Correlation')
plt.title('Pairwise Pearson Correlation Heatmap')
plt.xlabel('B Time Step')
plt.ylabel('A Time Step')
# plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
# plot PCA given existing PCA
plot_output_embeddings_pca_3d(trajectory_tokens, pca=pca, title="3D PCA of Output Embeddings")
# animate_pca_trajectory(output_embeddings, pca=pca)

In [ ]:
# Animate Trajectory
animate_pca_trajectory(stacked_output_embeddings_soft[:, 600:1200], pca=pca_outputs)

In [ ]:
# prompt: plot a 1 dimensional torch Tensor
embedding_variable = 0
plt.figure(figsize=(10, 4))
plt.stem(trajectory_tokens[0, :, embedding_variable].detach().cpu().numpy()) # Assuming batch size is 1
plt.title(f'Output Embedding {embedding_variable} Over Sequence Length')
plt.xlabel('Sequence Position')
plt.ylabel('Embedding Value')
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.stem(F.cosine_similarity(input_embeddings[0, 60], input_embeddings[0], dim=1).detach().cpu().numpy()) # Assuming batch size is 1
plt.title('Cosine Similarity to input_embeddings[60] Over Sequence Length')
plt.xlabel('Sequence Position')
plt.ylabel('Cosine Similarity')
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.stem(torch.pow(torch.abs(input_embeddings[0]), 2).sum(dim=1).detach().cpu().numpy()) # Assuming batch size is 1
plt.title('Input Embeddings Norm Over Generated Steps')
plt.xlabel('Generation Step')
plt.ylabel('Embedding Norm')
plt.grid(True)
plt.show()

In [ ]:
# prompt: plot the histogram of a list of integers

# Assuming 'data_list' is the list of integers you want to plot a histogram of
data_list = input_embeddings_hard[0, 100:, 10] # Example list

plt.figure(figsize=(8, 6))
plt.hist(data_list, bins='auto', edgecolor='black') # 'auto' chooses a reasonable number of bins
plt.title('Histogram of Second Embedding Dimension - Soft Mode')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()


In [ ]:
short_val_blocks = np.load("short_sequences_validation.npy")
short_val_dataset = DyckDataset(short_val_blocks)
short_val_loader = DataLoader(short_val_dataset, batch_size=128, shuffle=False, drop_last=True)


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
val_loss, val_acc = validate(model, short_val_loader, criterion, CONFIG['device'])
print(f"Val Loss: {val_loss} | Val Acc: {val_acc}")

In [ ]:
def generate_ellipse_points(
    a, b,
    center=(0.0, 0.0),
    step_size=0.1,
    num_rounds=1,
    start_angle=0.0
):
    """
    Generates 2D points along an ellipse and returns them as a torch Tensor.

    Parameters:
    - a: Semi-major axis length
    - b: Semi-minor axis length
    - center: Tuple (cx, cy) for ellipse center
    - step_size: Increment in radians along the ellipse
    - num_rounds: Number of full revolutions around the ellipse
    - start_angle: Starting angle in radians

    Returns:
    - points: torch.Tensor of shape [N, 2]
    """
    cx, cy = center

    total_angle = 2 * np.pi * num_rounds
    angles = np.arange(start_angle, start_angle + total_angle, step_size)

    x_points = cx + a * np.cos(angles)
    y_points = cy + b * np.sin(angles)

    points = np.stack((x_points, y_points), axis=1)

    return torch.from_numpy(points).float()

# Example usage:
if __name__ == "__main__":
    points = generate_ellipse_points(
        a=5,
        b=3,
        center=(0, 0),
        step_size=0.1,
        num_rounds=3
    )
    print(points.shape)  # Should be [N, 2]
    print(points[:5])  # Print first few points as a check


In [ ]:
points1 = generate_ellipse_points(
        a=5,
        b=3,
        center=(0, 0),
        step_size=0.1,
        num_rounds=3
    )

points2 = generate_ellipse_points(
        a=13,
        b=1,
        center=(0, 0),
        step_size=5,
        num_rounds=100
    )

points1 = torch.cat((points1, torch.zeros(points1.shape[0],1)), dim=1)
points2 = torch.cat((torch.zeros(points2.shape[0],1), points2), dim=1)

In [ ]:
import torch

def cca(X, Y, reg=1e-5):
    """
    X: [T, Dx]
    Y: [T, Dy]
    Returns: canonical correlations, Wx, Wy
    """
    T = X.shape[0]

    # Center
    X -= X.mean(dim=0, keepdim=True)
    Y -= Y.mean(dim=0, keepdim=True)

    # Covariance
    Cxx = X.T @ X / (T - 1) + reg * torch.eye(X.shape[1])
    Cyy = Y.T @ Y / (T - 1) + reg * torch.eye(Y.shape[1])
    Cxy = X.T @ Y / (T - 1)

    # Cholesky or inverse sqrt
    invCxx = torch.linalg.inv(Cxx)
    invCyy = torch.linalg.inv(Cyy)

    # Solve: eigenvalues of (invCxx @ Cxy @ invCyy @ Cyx)
    K = invCxx @ Cxy @ invCyy @ Cxy.T

    # Eigendecomposition
    eigvals, Wx = torch.linalg.eigh(K)
    eigvals = torch.clamp(eigvals, min=0.0)  # Avoid negatives due to num. issues
    canonical_corrs = torch.sqrt(eigvals)

    # Sort descending
    idx = torch.argsort(canonical_corrs, descending=True)
    canonical_corrs = canonical_corrs[idx]
    Wx = Wx[:, idx]

    # Get Wy
    Wy = invCyy @ Cxy.T @ Wx
    Wy = Wy / torch.norm(Wy, dim=0)

    return canonical_corrs, Wx, Wy

# Example usage:
X = stacked_input_embeddings_hard.squeeze()
Y = stacked_input_embeddings_raw.squeeze()

corrs, Wx, Wy = cca(X, Y)
print("Canonical correlations:", corrs)


In [ ]:
import torch

def cca_svd(X, Y, reg=1e-5):
    """
    Numerically stable CCA using whitening + SVD.
    """
    T = X.shape[0]

    # Center
    X = X - X.mean(dim=0, keepdim=True)
    Y = Y - Y.mean(dim=0, keepdim=True)

    # Covariance
    Cxx = X.T @ X / (T - 1) + reg * torch.eye(X.shape[1])
    Cyy = Y.T @ Y / (T - 1) + reg * torch.eye(Y.shape[1])

    # Whitening (inverse sqrt)
    Ux, Sx, _ = torch.linalg.svd(Cxx)
    Uy, Sy, _ = torch.linalg.svd(Cyy)

    invsqrtCxx = Ux @ torch.diag(1.0 / torch.sqrt(Sx)) @ Ux.T
    invsqrtCyy = Uy @ torch.diag(1.0 / torch.sqrt(Sy)) @ Uy.T

    # Cross-covariance
    Cxy = X.T @ Y / (T - 1)

    # Correlation matrix
    M = invsqrtCxx @ Cxy @ invsqrtCyy

    # Singular values = canonical correlations
    U, S, Vh = torch.linalg.svd(M)

    return S, U, Vh.T

# Example usage:
X = stacked_input_embeddings_hard[:, 200:].squeeze()
Y = stacked_input_embeddings_raw[:, 200:].squeeze()

corrs, Wx, Wy = cca_svd(X, Y)
print("Canonical correlations:", corrs)

In [ ]:
### Autoencoder Tests
train_tensor = torch.tensor(train_blocks, dtype=torch.long)
val_tensor = torch.tensor(val_blocks, dtype=torch.long)
offset = 4
x = val_tensor[offset-1:offset].to(CONFIG['device'])

logits, _, _, _ = model(x,x[:,:-1])

probs = F.softmax(logits, dim=-1)

print(f"True: {''.join([id2token[s.item()] for s in x[0]])}")
print(f"Pred: {''.join([id2token[s.argmax().item()] for s in probs[0]])}")

plot_token_distributions(probs, CONFIG['vocab'])

In [ ]:
### Data Preperation for Autoencoder
def create_stacked_trajectories_array(initial_seqs, model, num_steps, device, generation_batch_size=1024):
    """
    Generates trajectories from initial sequences using the model.
    
    Args:
        initial_seqs (list): List of initial sequences.
        model: The model to use for generation.
        num_steps (int): Number of steps to generate.
        device: Device to run the model on.
        generation_batch_size (int): Size of each batch for generation.
    
    Returns:
        list: List of generated trajectories.
    """

    # filter only sequences that are long enough
    seqs_filtered = [seq[:(CONFIG['context_window'] - 1)] for seq in initial_seqs if len(seq) >= (CONFIG['context_window'] - 1)]
    n_batches = len(seqs_filtered) // generation_batch_size

    trajectories_list = []
    for batch in tqdm(range(n_batches)):
        x = seqs_filtered[(batch * generation_batch_size):((batch + 1) * generation_batch_size)]
        _, _, _, generated_tokens = soft_generate_batch(model, x, num_steps, device, mode="hard")
        trajectories_list.extend(generated_tokens)

    # encode the trajectories into a np.uint8 array
    trajectories = np.zeros((len(trajectories_list), len(trajectories_list[0])), dtype=np.uint8)

    for i, seq in enumerate(tqdm(trajectories_list)):
        for j, token in enumerate(seq):
            trajectories[i, j] = token2id[token]

    # prepend BOS token to each trajectory
    trajectories = np.insert(trajectories, 0, bos_id, axis=1)  # Insert BOS token at the beginning of each trajectory

    # Stack context windows
    trajectories_stacked = stack_context_windows_tokens(trajectories, context_window=CONFIG['context_window'])  # [trajectories, time, context_window]

    return trajectories_stacked


num_steps = 100

train_trajectories_stacked = create_stacked_trajectories_array(
    initial_seqs=train_seqs,
    model=model,
    num_steps=num_steps,
    device=CONFIG['device'],
    generation_batch_size=1024
)

val_trajectories_stacked = create_stacked_trajectories_array(
    initial_seqs=val_seqs,
    model=model,
    num_steps=num_steps,
    device=CONFIG['device'],
    generation_batch_size=1024
)


In [ ]:
# Save the trajectories to a HF dataset
from datasets import Dataset, DatasetInfo
train_trajectories_dataset = Dataset.from_dict(
    {"trajectory_ids": train_trajectories_stacked},
    #  dataset_info=DatasetInfo(
    #     description="Dyck2 Trajectories Dataset",
    #     features={
    #         "token2id": token2id,
    #         "id2token": id2token,
    #         "num_steps": num_steps,
    #         "config": CONFIG
    #     }
    # ),
    split="train"
)
train_trajectories_dataset.push_to_hub("Gal-Kinberg/LanguageDynamics", split="train", private=True)

val_trajectories_dataset = Dataset.from_dict(
    {"trajectory_ids": val_trajectories_stacked},
    # dataset_info=DatasetInfo(
    #     description="Dyck2 Trajectories Dataset",
    #     features={
    #         "token2id": token2id,
    #         "id2token": id2token,
    #         "num_steps": num_steps,
    #         "config": CONFIG
    #     }
    # ),
    split="validation"
)
val_trajectories_dataset.push_to_hub("Gal-Kinberg/LanguageDynamics", split="validation", private=True)



In [8]:
from datasets import load_dataset
train_trajectories_dataset_loaded = load_dataset("Gal-Kinberg/LanguageDynamics", split="train").with_format("numpy")
val_trajectories_dataset_loaded = load_dataset("Gal-Kinberg/LanguageDynamics", split="validation").with_format("numpy")

In [ ]:
train_trajectories_np = train_trajectories_dataset_loaded['trajectory_ids']
val_trajectories_np = val_trajectories_dataset_loaded['trajectory_ids']

In [ ]:
class TrajectoryDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]
    
train_trajectories_dataset_np = TrajectoryDataset(train_trajectories_np)
val_trajectories_dataset_np = TrajectoryDataset(val_trajectories_np)

In [ ]:
def collate_fn_np(batch):
    # 'batch' is a list of np arrays, of shape [T, C] of trajectory IDs
    trajs = np.array(batch)
    trajs = torch.tensor(trajs, dtype=torch.long)  # shape [B, T, C]
    return trajs

KAE_train_loader = DataLoader(train_trajectories_dataset_np, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn_np, drop_last=True)
KAE_val_loader = DataLoader(val_trajectories_dataset_np, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn_np, drop_last=True)


In [ ]:
# def collate_fn(batch):
#     #TODO - write better loading  
#     # 'batch' is a list of dicts, each with a key 'trajectory_ids' containing a numpy array of shape [T, C] of trajectory IDs
#     trajs = np.array([item["trajectory_ids"] for item in batch])
#     trajs = torch.tensor(trajs, dtype=torch.long)  # shape [B, T, C]
#     return trajs

# KAE_train_loader = DataLoader(train_trajectories_dataset_loaded, batch_size=CONFIG['batch_size'], shuffle=True, collate_fn=collate_fn, drop_last=True)
# KAE_val_loader = DataLoader(val_trajectories_dataset_loaded, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate_fn, drop_last=True)


In [10]:
class KoopmanNet(nn.Module):
    def __init__(self, obsdim: int, n_diagonals: int = 2):
        super().__init__()
        self.obsdim = obsdim
        
        # Initialize diagonal parameters - linearly spaced from 1 to 0
        self.kMatrixDiag = nn.Parameter(torch.linspace(1, 0, self.obsdim))
        
        # Get upper triangular indices for off-diagonal elements
        rows, cols = torch.triu_indices(self.obsdim, self.obsdim, offset=1)
        
        # Only keep the first two off-diagonals
        mask = cols - rows <= n_diagonals
        self.register_buffer('row_idx', rows[mask])
        self.register_buffer('col_idx', cols[mask])
        
        # Initialize off-diagonal parameters
        self.kMatrixUT = nn.Parameter(0.1 * torch.rand(self.row_idx.size(0)))
        self.kMatrixLT = nn.Parameter(0.1 * torch.rand(self.row_idx.size(0)))

    def get_koopman_matrix(self) -> torch.Tensor:
        """Constructs the Koopman matrix from diagonal and off-diagonal parameters."""
        # Initialize zero matrix
        K = torch.zeros(self.obsdim, self.obsdim, device=self.kMatrixDiag.device)
        
        # Set diagonal elements
        K.diagonal().copy_(self.kMatrixDiag)
        
        # Set off-diagonal elements (upper triangle)
        K[self.row_idx, self.col_idx] = self.kMatrixUT
        # Set off-diagonal elements (lower triangle)
        K[self.col_idx, self.row_idx] = self.kMatrixLT
        
        return K

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Applies the Koopman operator to the observables."""
        K = self.get_koopman_matrix()
        z_predicted = torch.matmul(z, K.T)
        self.K = K  # Store the Koopman matrix for later use
        return z_predicted

    @property
    def koopmanMatrix(self, requires_grad: bool = True) -> torch.Tensor:
        """Returns the current Koopman matrix."""
        if requires_grad:
            return self.K
        else:
            return self.K.detach()

In [13]:
class TransformerKoopmanAutoencoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id, sos_id, n_latents=8, latent_dropout=0.03, n_diagonals=5):
        super().__init__()
        self.encoder = TransformerEncoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, cls_id)
        self.decoder = TransformerDecoder(vocab_size, embed_dim, latent_dim, n_layers, n_heads, ffn_dim, context_window, sos_id, n_latents)
        self.koopman = KoopmanNet(obsdim=latent_dim, n_diagonals=n_diagonals)
        self.latent_dim = latent_dim
        self.latent_dropout = nn.Dropout(latent_dropout)

        self.cls_id = cls_id
        self.sos_id = sos_id
        self.context_window = context_window

        # weight tying of embeddings and head
        self.decoder.embed.weight = self.encoder.embed.weight
        self.decoder.head.weight = self.encoder.embed.weight

    def forward(self, x, decoding_seed, return_internals=True):
        B, T = x.shape
        latent, initial_embeddings, final_embeddings = self.encoder(x, return_internals=return_internals)
        logits, final_embeddings = self.decoder(decoding_seed, self.latent_dropout(latent), return_internals=return_internals)

        if return_internals:
            return logits, latent, initial_embeddings, final_embeddings
        else:
            return logits, latent

In [14]:
### Training Parameters
reconstruction_coef = 10.0
koopman_coef = 1.0
regularization_coef = 0.01
teacher_forcing = False
do_validation = True

In [20]:
### Koopman Autoencoder Training
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_koopman_losses = []
        self.train_regularization_losses = []
        self.train_accuracies = []
        self.train_koopman_accuracies = []
        
        # Validation metrics
        self.val_reconstruction_losses = []
        self.val_koopman_losses = []
        self.val_accuracies = []
        self.val_koopman_accuracies = []
        
        self.epochs = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict['train_total_losses']
        self.train_reconstruction_losses = metrics_dict['train_reconstruction_losses']
        self.train_koopman_losses = metrics_dict['train_koopman_losses']
        self.train_regularization_losses = metrics_dict['train_regularization_losses']
        self.train_accuracies = metrics_dict['train_accuracies']
        try:
            self.train_koopman_accuracies = metrics_dict['train_koopman_accuracies']
        except KeyError:
            self.train_koopman_accuracies = []
        self.val_reconstruction_losses = metrics_dict['val_reconstruction_losses']
        self.val_koopman_losses = metrics_dict['val_koopman_losses']
        self.val_accuracies = metrics_dict['val_accuracies']
        try:
            self.val_koopman_accuracies = metrics_dict['val_koopman_accuracies']
        except KeyError:
            self.val_koopman_accuracies = []
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction and Koopman losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        axes[0,0].plot(self.epochs, self.train_koopman_losses, 
                    color=train_color, linestyle='--', label='Train Koopman')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        axes[0,0].plot(self.epochs, self.val_koopman_losses, 
                    color=val_color, linestyle='--', label='Val Koopman')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction and Koopman Losses')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: Accuracies
        axes[0,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        try:
            axes[0,1].plot(self.epochs, self.train_koopman_accuracies, 
                        color=train_color, linestyle='--', label='Train Koopman')
        except :
            pass  # If train_koopman_accuracies is not available, skip this line
        axes[0,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        try:
            axes[0,1].plot(self.epochs, self.val_koopman_accuracies, 
                        color=val_color, linestyle='--', label='Val Koopman')
        except:
            pass  # If val_koopman_accuracies is not available, skip this line
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Accuracy')
        axes[0,1].set_title('Model Accuracies')
        axes[0,1].legend()
        axes[0,1].grid(True)

        # Plot 3: Total training loss
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color)  # Green
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        axes[1,0].set_title('Total Training Loss')
        axes[1,0].grid(True)

        # Plot 4: Regularization loss
        axes[1,1].plot(self.epochs, self.train_regularization_losses, color=train_color)  # Orange
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Loss')
        axes[1,1].set_title('Training Regularization Loss')
        axes[1,1].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, device, optimizer=None):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch and loaded metrics
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    for key in ['vocab_size', 'embed_dim', 'n_layers', 'n_heads', 'ffn_dim', 'context_window']:
        if CONFIG[key] != saved_config[key]:
            raise ValueError(f"Checkpoint config mismatch for {key}: "
                           f"current={CONFIG[key]}, saved={saved_config[key]}")

    starting_epoch = checkpoint['epoch']
    return starting_epoch, checkpoint['metrics']

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if CONFIG['mode'] == 'KAE':
        model = TransformerKoopmanAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents'],
            n_diagonals=CONFIG['n_diagonals']
        ).to(CONFIG['device'])

    else:
        raise ValueError(f"Unsupported mode: {CONFIG['mode']}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'])
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, CONFIG['model_save_prefix'] + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    if CONFIG['checkpoint_path']:
        try:
            starting_epoch, saved_metrics = load_checkpoint(
                CONFIG['checkpoint_path'],
                model,
                CONFIG['device'],
                optimizer
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = CONFIG['lr']
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0

    print("Starting training loop...")
    global_start = time.time()

    def validate(model, val_loader, criterion, device):
        model.eval()
        total_reconstruction_loss = 0
        total_koopman_loss = 0
        total_acc = 0
        total_koopman_acc = 0
        with torch.no_grad():
            for x in val_loader:
                # x is [B, T, C] tokens as tensors
                B, T, C = x.shape
                # reshape to [B*T, C] for model input
                x = x.view(B * T, C)
                x = x.to(CONFIG['device'])
                y = x.clone() # for the masked reconstruction loss

                # Compute the reconstruction loss
                logits, latent, _, _ = model(x, decoding_seed=x[:,:-1]) # logits: [B*T, C, vocab_size], latent: [B*T, latent_dim]
                loss_reconstruction = criterion(logits.view(-1, CONFIG['vocab_size']), x.view(-1))

                if teacher_forcing:
                    # advance all latents by one step using the koopman operator
                    latent_pred = model.koopman(latent)  # [B*T, latent_dim]
                
                else:
                    # initialize latent predictions
                    latent_pred = torch.zeros((B*T, model.latent_dim), device=CONFIG['device'])
                    # get the first latent from each trajectory
                    curr_latent = latent[::T, :]  # [B, latent_dim]
                    # advance the first latent T timesteps using the koopman operator
                    for t in range(T):
                        curr_latent = model.koopman(curr_latent)  # [B, latent_dim]
                        # store the latent prediction for this timestep
                        latent_pred[t::T] = curr_latent
                
                # decode the latent predictions
                logits_koopman, _ = model.decoder(x[1:,:-1], model.latent_dropout(latent_pred[:-1]), return_internals=True)  # [B*T - 1, C, vocab_size]
                y[::T] = torch.full([C], fill_value=pad_id, dtype=torch.long)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)

                # y[::T].fill_(pad_id)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)
                loss_koopman = criterion(logits_koopman.view(-1, CONFIG['vocab_size']), y[1:].view(-1))

                acc = calculate_accuracy(logits, x, pad_id)  # autoencoding reconstruction accuracy
                koopman_acc = calculate_accuracy(logits_koopman, y[1:], pad_id)  # koopman accuracy

                total_reconstruction_loss += loss_reconstruction.item()
                total_koopman_loss += loss_koopman.item()
                total_acc += acc
                total_koopman_acc += koopman_acc
        return total_reconstruction_loss / len(val_loader), total_koopman_loss / len(val_loader), total_acc / len(val_loader), total_koopman_acc / len(val_loader)

    # Training loop
    for epoch in range(starting_epoch, CONFIG['epochs']):
        model.train()
        epoch_loss = 0.0
        epoch_loss_reconstruction = 0.0
        epoch_loss_koopman = 0.0
        epoch_loss_regularization = 0.0
        epoch_acc = 0.0
        epoch_koopman_acc = 0.0
        start = time.time()

        # Training phase
        for batch, x in enumerate(tqdm(KAE_train_loader, desc=f"Epoch {epoch+1} Training")):
            # x is [B, T, C] tokens as tensors
            B, T, C = x.shape
            # reshape to [B*T, C] for model input
            x = x.view(B * T, C)
            x = x.to(CONFIG['device'])
            y = x.clone() # for the masked reconstruction loss
            
            logits, latent, _, _ = model(x, decoding_seed=x[:,:-1]) # logits: [B*T, C, vocab_size], latent: [B*T, latent_dim]
            loss_reconstruction = criterion(logits.view(-1, CONFIG['vocab_size']), x.view(-1))

            if teacher_forcing:
                # advance all latents by one step using the koopman operator
                latent_pred = model.koopman(latent)  # [B*T, latent_dim]
            
            else:
                # initialize latent predictions
                latent_pred = torch.zeros((B*T, model.latent_dim), device=CONFIG['device'])
                # get the first latent from each trajectory
                curr_latent = latent[::T, :]  # [B, latent_dim]
                # advance the first latent T timesteps using the koopman operator
                for t in range(T):
                    curr_latent = model.koopman(curr_latent)  # [B, latent_dim]
                    # store the latent prediction for this timestep
                    latent_pred[t::T] = curr_latent
            
            # decode the latent predictions
            logits_koopman, _ = model.decoder(x[1:,:-1], model.latent_dropout(latent_pred[:-1]), return_internals=True)  # [B*T - 1, C, vocab_size]
            y[::T] = torch.full([C], fill_value=pad_id, dtype=torch.long)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)

            # y[::T].fill_(pad_id)  # Set the first window of each trajectory to pad_id to ignore it in the loss calculation (it cannot be predicted)
            loss_koopman = criterion(logits_koopman.view(-1, CONFIG['vocab_size']), y[1:].view(-1))
                
            regularization_loss = torch.sum(torch.pow(model.koopman.koopmanMatrix, 2))

            loss = reconstruction_coef * loss_reconstruction + koopman_coef * loss_koopman + regularization_coef * regularization_loss
            acc = calculate_accuracy(logits, x, pad_id)  # autoencoding reconstruction accuracy
            koopman_acc = calculate_accuracy(logits_koopman, y[1:], pad_id)  # koopman accuracy

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_loss_reconstruction += loss_reconstruction.item()
            epoch_loss_koopman += loss_koopman.item()
            epoch_loss_regularization += regularization_loss.item()
            epoch_acc += acc
            epoch_koopman_acc += koopman_acc

        # Validation phase
        val_reconstruction_loss, val_koopman_loss, val_acc, val_koopman_acc = validate(model, KAE_val_loader, criterion, CONFIG['device'])

        # Calculate average losses and accuracies
        train_loss = epoch_loss / len(KAE_train_loader)
        train_acc = epoch_acc / len(KAE_train_loader)
        train_koopman_acc = epoch_koopman_acc / len(KAE_train_loader)
        train_loss_reconstruction = epoch_loss_reconstruction / len(KAE_train_loader)
        train_loss_koopman = epoch_loss_koopman / len(KAE_train_loader)
        train_loss_regularization = epoch_loss_regularization / len(KAE_train_loader)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_loss_reconstruction)
        metrics.train_koopman_losses.append(train_loss_koopman)
        metrics.train_regularization_losses.append(train_loss_regularization)
        metrics.train_accuracies.append(train_acc)
        metrics.train_koopman_accuracies.append(train_koopman_acc)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_koopman_losses.append(val_koopman_loss)
        metrics.val_accuracies.append(val_acc)
        metrics.val_koopman_accuracies.append(val_koopman_acc)

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Total Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train Koopman Acc: {train_koopman_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_loss_reconstruction:.4f} | Train Koopman Loss: {train_loss_koopman:.4f} | Train Regularization Loss: {train_loss_regularization:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val Koopman Loss: {val_koopman_loss:.4f} | Val Acc: {val_acc:.4f} | Val Koopman Acc: {val_koopman_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")

        # Plot metrics
        metrics.plot_metrics(save_dir)

        # Save checkpoint
        if (epoch+1) % CONFIG['save_every'] == 0 or (epoch+1) == CONFIG['epochs']:
            # ckpt_path = save_dir / f"{CONFIG['model_save_prefix']}_trial_{trial+1}_epoch{epoch+1}.pt"
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': CONFIG,
                'epoch': epoch+1,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_koopman_losses': metrics.train_koopman_losses,
                    'train_regularization_losses': metrics.train_regularization_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'train_koopman_accuracies': metrics.train_koopman_accuracies,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_koopman_losses': metrics.val_koopman_losses,
                    'val_accuracies': metrics.val_accuracies,
                    'val_koopman_accuracies': metrics.val_koopman_accuracies
                }
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

Starting training loop...


Epoch 1 Training:   0%|          | 0/1648 [00:00<?, ?it/s]

Epoch 1 Training:   0%|          | 0/1648 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 202.00 MiB. GPU 0 has a total capacity of 10.90 GiB of which 9.88 MiB is free. Process 3269713 has 1.25 GiB memory in use. Process 3274547 has 1.25 GiB memory in use. Process 3288310 has 1.25 GiB memory in use. Process 3294225 has 1.25 GiB memory in use. Including non-PyTorch memory, this process has 5.88 GiB memory in use. Of the allocated memory 5.62 GiB is allocated by PyTorch, and 107.82 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [23]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [21]:
del(model, x, y, logits, latent, optimizer, criterion, latent_pred, logits_koopman, loss_reconstruction, loss_koopman, regularization_loss, loss)

NameError: name 'logits' is not defined

In [ ]:
len(KAE_val_loader)

In [22]:
del(loss_reconstruction)